# PDAC detection and segmentation in CT (PANORAMA)

Two-stage pipeline: a CT-only 3D pancreas localizer, then nnU-Net (Residual Encoder U-Net, ResEnc-M) segmentation of pancreas and PDAC inside a fixed ROI crop. Trained and evaluated on the full public PANORAMA cohort (2,238 contrast-enhanced CT studies) with a locked patient-level split.

- Runs on Kaggle, Google Colab or a local machine. Every phase is resumable and stores its results in a private Hugging Face repository (`WORK_REPO` in cell 1).
- Set `WORK_REPO` and provide a Hugging Face **write** token through an `HF_TOKEN` environment variable or a Kaggle/Colab secret before running.
- The outputs below are from the final run (21 Oct 2026). See the repository README and `docs/PDAC_Segmentation_Report.pdf` for the method and results.


In [1]:
# ============================================================
# CELL 1 — CONFIG (the only cell you edit)
# ============================================================
SOURCE_REPO = "MedOtter/PANORAMA"                  # public dataset (read-only)
WORK_REPO = "<your-hf-username>/pdac-progress"     # your PRIVATE Hugging Face repo, used as durable storage
WORK_REPO_TYPE = "model"
SEED = 20261018

# Optional fallback token. Prefer an HF_TOKEN environment variable or a Kaggle/Colab secret.
# Never commit a real token.
HF_TOKEN_FALLBACK = ""

print("Config loaded.")

Config loaded.


In [2]:
# ============================================================
# CELL 2 — PLATFORM, PATHS AND HARDWARE PROBE
# Works on Kaggle, Colab and local (VS Code). Safe to re-run.
# ============================================================
import os, sys, json, time, shutil, socket, platform, subprocess, importlib.util
from pathlib import Path
from datetime import datetime, timezone

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # keeps outputs short

def detect_platform():
    if Path("/kaggle/working").exists():
        return "kaggle"
    if importlib.util.find_spec("google.colab") is not None:
        return "colab"
    return "local"

PLATFORM = detect_platform()

if PLATFORM == "kaggle":
    WORK = Path("/kaggle/working/pdac")
    scratch_candidates = [Path("/kaggle/temp"), Path("/tmp"), Path("/kaggle/working")]
elif PLATFORM == "colab":
    WORK = Path("/content/pdac")
    scratch_candidates = [Path("/content"), Path("/tmp")]
else:
    WORK = Path.home() / "pdac_work"
    scratch_candidates = [Path.home() / "pdac_work", Path("/tmp")]

WORK.mkdir(parents=True, exist_ok=True)

disk_rows = []
for p in scratch_candidates:
    try:
        p.mkdir(parents=True, exist_ok=True)
        u = shutil.disk_usage(p)
        disk_rows.append({"path": str(p), "free_gb": round(u.free / 1e9, 1),
                          "total_gb": round(u.total / 1e9, 1), "device": os.stat(p).st_dev})
    except Exception as e:
        disk_rows.append({"path": str(p), "error": str(e)})

usable = [r for r in disk_rows if "free_gb" in r]
best = max(usable, key=lambda r: r["free_gb"])
SCRATCH = Path(best["path"]) / "pdac_scratch"
SCRATCH.mkdir(parents=True, exist_ok=True)

# ---- hardware ----
cpu_count = os.cpu_count()
try:
    cpu_affinity = len(os.sched_getaffinity(0))
except Exception:
    cpu_affinity = cpu_count
try:
    import psutil
    ram_gb = round(psutil.virtual_memory().total / 1e9, 1)
except Exception:
    ram_gb = None

gpu = {"cuda": False}
try:
    import torch
    gpu["torch"] = torch.__version__
    gpu["cuda"] = torch.cuda.is_available()
    if gpu["cuda"]:
        gpu["name"] = torch.cuda.get_device_name(0)
        gpu["count"] = torch.cuda.device_count()
        gpu["mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    gpu["mps"] = bool(getattr(torch.backends, "mps", None) and torch.backends.mps.is_available())
except Exception as e:
    gpu["torch_error"] = str(e)

try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
    internet = True
except Exception:
    internet = False

ENV = {
    "platform": PLATFORM, "python": sys.version.split()[0], "os": platform.platform(),
    "cpu_count": cpu_count, "cpu_affinity": cpu_affinity, "ram_gb": ram_gb,
    "gpu": gpu, "internet": internet, "work": str(WORK), "scratch": str(SCRATCH),
    "disks": disk_rows, "time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(WORK / "env_probe.json").write_text(json.dumps(ENV, indent=2))

print("=" * 60)
print(f"Platform     : {PLATFORM}")
print(f"Python       : {ENV['python']}")
print(f"CPUs         : {cpu_count} (usable {cpu_affinity}) | RAM: {ram_gb} GB")
print(f"GPU          : {gpu}")
print(f"Internet     : {internet}")
print(f"WORK         : {WORK}")
print(f"SCRATCH      : {SCRATCH}")
print("Disks:")
for r in disk_rows:
    print("   ", r)
print("=" * 60)
if not internet:
    print("✗ No internet. On Kaggle: Settings → Internet → On (needs phone verification).")

Platform     : kaggle
Python       : 3.13.15
CPUs         : 4 (usable 4) | RAM: 33.7 GB
GPU          : {'cuda': True, 'torch': '2.11.0+cu128', 'name': 'Tesla T4', 'count': 2, 'mem_gb': 15.6, 'mps': False}
Internet     : True
WORK         : /kaggle/working/pdac
SCRATCH      : /kaggle/temp/pdac_scratch
Disks:
    {'path': '/kaggle/temp', 'free_gb': 1146.0, 'total_gb': 8758.4, 'device': 54}
    {'path': '/tmp', 'free_gb': 1146.0, 'total_gb': 8758.4, 'device': 54}
    {'path': '/kaggle/working', 'free_gb': 20.9, 'total_gb': 21.0, 'device': 1794}


In [3]:
# ============================================================
# CELL 3 — INSTALL ONLY MISSING PACKAGES + VERIFY nnU-Net
# ============================================================
import importlib, inspect
from importlib.metadata import version, PackageNotFoundError

REQUIRED = {  # import name : pip name
    "SimpleITK": "SimpleITK",
    "nibabel": "nibabel",
    "huggingface_hub": "huggingface_hub",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "psutil": "psutil",
    "nnunetv2": "nnunetv2",
}

torch_before = None
try:
    torch_before = version("torch")
except PackageNotFoundError:
    pass

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                       capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("pip install failed — see the output above.")
else:
    print("Nothing to install.")
importlib.invalidate_caches()

print("\nVersions:")
for pip_name in ["torch", "numpy", "SimpleITK", "nibabel", "huggingface_hub", "pandas",
                 "scikit-learn", "nnunetv2", "dynamic_network_architectures", "batchgeneratorsv2", "blosc2"]:
    try:
        print(f"  {pip_name:30s} {version(pip_name)}")
    except PackageNotFoundError:
        print(f"  {pip_name:30s} (not installed)")

torch_after = version("torch")
if torch_before and torch_after != torch_before:
    print(f"\n⚠ torch changed {torch_before} → {torch_after}. RESTART the session, then re-run cells 1–3.")

# Verify the nnU-Net trainer hooks that our resumable trainer will rely on
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
src = inspect.getsource(nnUNetTrainer)
for attr in ["self.num_epochs", "self.save_every", "self.num_iterations_per_epoch",
             "def on_epoch_end", "def save_checkpoint", "def load_checkpoint", "checkpoint_latest.pth"]:
    print(f"  nnU-Net has {attr:32s}: {attr in src}")
print("✓ nnU-Net import OK")

Installing: ['nnunetv2']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 5.1 MB/s eta 0:00:00


Versions:
  torch                          2.11.0+cu128
  numpy                          2.1.3
  SimpleITK                      2.5.6
  nibabel                        5.4.2
  huggingface_hub                1.29.0
  pandas                         2.3.3
  scikit-learn

In [4]:
# ============================================================
# CELL 4 — PERSISTENT STORE (your private Hugging Face repo)
# Everything that must survive a crash / new session / new platform lives here.
# Writes ONLY to WORK_REPO from Cell 1.
# ============================================================
from huggingface_hub import HfApi, hf_hub_download
try:
    from huggingface_hub.errors import EntryNotFoundError, RepositoryNotFoundError
except ImportError:
    from huggingface_hub.utils import EntryNotFoundError, RepositoryNotFoundError

def get_hf_token():
    """Order: env var → Kaggle secret → Colab secret → cached login → Cell 1 fallback."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok, "environment variable"
    if PLATFORM == "kaggle":
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
            if tok:
                return tok, "Kaggle secret"
        except Exception:
            pass
    if PLATFORM == "colab":
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok, "Colab secret"
        except Exception:
            pass
    try:
        from huggingface_hub import get_token
        tok = get_token()
        if tok:
            return tok, "cached HF login"
    except Exception:
        pass
    fb = globals().get("HF_TOKEN_FALLBACK", "")
    if fb and fb.startswith("hf_"):
        return fb, "Cell 1 fallback"
    return None, None

HF_TOKEN, TOKEN_SOURCE = get_hf_token()
if not HF_TOKEN:
    raise RuntimeError("No HF token found. Paste it into HF_TOKEN_FALLBACK in Cell 1, or add a secret named HF_TOKEN.")
os.environ["HF_TOKEN"] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
me = api.whoami()
role = (me.get("auth", {}).get("accessToken", {}) or {}).get("role", "unknown")
print(f"Logged in to Hugging Face as: {me['name']} | token role: {role} | source: {TOKEN_SOURCE}")
if role == "read":
    raise RuntimeError("Token is READ-only. Use a WRITE token.")

owner = WORK_REPO.split("/")[0]
if me["name"] != owner and owner not in [o.get("name") for o in me.get("orgs", [])]:
    raise RuntimeError(f"Token belongs to '{me['name']}', not to the repo owner '{owner}'.")
try:
    info = api.repo_info(WORK_REPO, repo_type=WORK_REPO_TYPE)
except RepositoryNotFoundError:
    raise RuntimeError(f"Repo {WORK_REPO} ({WORK_REPO_TYPE}) not found — check Cell 1.")
if not info.private:
    raise RuntimeError(f"{WORK_REPO} is PUBLIC. Make it private before storing anything.")


class Store:
    """Tiny wrapper around ONE private HF repo used as durable storage."""

    def __init__(self, api, repo_id, repo_type, cache_dir):
        self.api, self.repo_id, self.repo_type = api, repo_id, repo_type
        self.cache = Path(cache_dir)
        self.cache.mkdir(parents=True, exist_ok=True)

    def exists(self, path):
        return self.api.file_exists(self.repo_id, path, repo_type=self.repo_type)

    def put(self, local_path, path, msg=None):
        self.api.upload_file(path_or_fileobj=str(local_path), path_in_repo=path,
                             repo_id=self.repo_id, repo_type=self.repo_type,
                             commit_message=msg or f"put {path}")

    def put_json(self, obj, path, msg=None):
        tmp = self.cache / ("_tmp_" + path.replace("/", "__"))
        tmp.write_text(json.dumps(obj, indent=2, default=str))
        self.put(tmp, path, msg)
        tmp.unlink(missing_ok=True)

    def get(self, path, force=False):
        try:
            return Path(hf_hub_download(self.repo_id, path, repo_type=self.repo_type,
                                        local_dir=str(self.cache), force_download=force,
                                        token=HF_TOKEN))
        except EntryNotFoundError:
            return None

    def get_json(self, path):
        p = self.get(path, force=True)
        return json.loads(p.read_text()) if p else None

    def list(self, prefix):
        try:
            return sorted(f.path for f in self.api.list_repo_tree(
                self.repo_id, path_in_repo=prefix, repo_type=self.repo_type, recursive=True)
                if hasattr(f, "size"))
        except Exception:
            return []

    # phase markers
    def done(self, phase):
        return self.exists(f"state/{phase}.done.json")

    def mark_done(self, phase, info=None):
        self.put_json({"phase": phase, "info": info or {}, "platform": PLATFORM,
                       "time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")},
                      f"state/{phase}.done.json", msg=f"DONE {phase}")


STORE = Store(api, WORK_REPO, WORK_REPO_TYPE, WORK / "store_cache")

# Round-trip test
beat = {"platform": PLATFORM, "time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "gpu": ENV["gpu"].get("name", "none")}
STORE.put_json(beat, "state/heartbeat.json", msg="heartbeat")
back = STORE.get_json("state/heartbeat.json")
assert back["time_utc"] == beat["time_utc"], "Round-trip mismatch"

done_phases = [p.split("/")[-1].replace(".done.json", "") for p in STORE.list("state") if p.endswith(".done.json")]
print(f"✓ Store ready: https://huggingface.co/{WORK_REPO}")
print("✓ Write/read round-trip OK")
print(f"Phases already completed: {done_phases if done_phases else 'none yet'}")

Logged in to Hugging Face as: <hf-user> | token role: fineGrained | source: Cell 1 fallback
✓ Store ready: https://huggingface.co/<your-hf-username>/pdac-progress
✓ Write/read round-trip OK
Phases already completed: ['localizer', 'nnunet_prep', 'passA', 'passB']


In [5]:
# ============================================================
# CELL 5 — DATASET INVENTORY + DOWNLOAD SPEED TEST
# Builds the case list once and stores it; later sessions reuse it.
# ============================================================
import pandas as pd

INV_PATH = "meta/inventory.csv"

if STORE.exists(INV_PATH):
    inventory = pd.read_csv(STORE.get(INV_PATH, force=True), dtype=str)
    print("Inventory loaded from store (locked).")
else:
    def tree(prefix):
        return [f for f in api.list_repo_tree(SOURCE_REPO, path_in_repo=prefix,
                                              repo_type="dataset", recursive=True)
                if hasattr(f, "size")]
    imgs = {f.path.split("/")[-1].replace("_0000.nii.gz", ""): f for f in tree("images")}
    man = {f.path.split("/")[-1].replace(".nii.gz", ""): f for f in tree("labels/manual")}
    aut = {f.path.split("/")[-1].replace(".nii.gz", ""): f for f in tree("labels/automatic")}
    clin = pd.read_csv(hf_hub_download(SOURCE_REPO, "clinical_information.csv",
                                       repo_type="dataset", local_dir=str(WORK / "src_meta")), dtype=str)
    rows = []
    for _, r in clin.iterrows():
        sid = r["PANORAMA_study_id"]
        lab = man.get(sid) or aut.get(sid)
        rows.append({
            "study_id": sid, "patient_id": r["PANORAMA_patient_id"], "label": r["label"],
            "tier": r["annotation_tier"], "level": r["level"],
            "external_source": r.get("external_source") if isinstance(r.get("external_source"), str) else "",
            "scanner": r.get("scanner"), "image_path": imgs[sid].path if sid in imgs else None,
            "image_mb": round(imgs[sid].size / 1e6, 2) if sid in imgs else None,
            "label_path": lab.path if lab else None,
            "in_manual": sid in man, "in_automatic": sid in aut,
        })
    inventory = pd.DataFrame(rows)
    inventory.to_csv(WORK / "inventory.csv", index=False)
    STORE.put(WORK / "inventory.csv", INV_PATH, msg="inventory")
    print("Inventory built and stored.")

inventory["image_mb"] = inventory["image_mb"].astype(float)
print(f"Cases: {len(inventory)} | missing image: {inventory.image_path.isna().sum()} | "
      f"missing label: {inventory.label_path.isna().sum()}")
print(f"Both manual+automatic label: {(inventory.in_manual.astype(str)=='True').sum() if 'in_manual' in inventory else 'n/a'} manual")
print(f"Total image size: {inventory.image_mb.sum()/1e3:.1f} GB | median {inventory.image_mb.median():.1f} MB | max {inventory.image_mb.max():.1f} MB")
print(pd.crosstab(inventory.tier, inventory.label))

# ---- speed test on one median-size case (downloaded to SCRATCH, then deleted) ----
import SimpleITK as sitk
probe = inventory.iloc[(inventory.image_mb - inventory.image_mb.median()).abs().argsort().iloc[0]]
probe_dir = SCRATCH / "probe"
t0 = time.time()
p = hf_hub_download(SOURCE_REPO, probe.image_path, repo_type="dataset", local_dir=str(probe_dir))
t_dl = time.time() - t0
t0 = time.time()
img = sitk.ReadImage(p)
t_read = time.time() - t0
print(f"\nSpeed test: {probe.study_id} ({probe.image_mb:.0f} MB) downloaded in {t_dl:.1f}s "
      f"→ {probe.image_mb / max(t_dl, 1e-6):.1f} MB/s | read in {t_read:.1f}s | "
      f"size {img.GetSize()} spacing {tuple(round(s, 3) for s in img.GetSpacing())}")
del img
shutil.rmtree(probe_dir, ignore_errors=True)
est_hours = inventory.image_mb.sum() / max(probe.image_mb / max(t_dl, 1e-6), 1e-6) / 3600
print(f"Rough single-stream estimate for all 194 GB: {est_hours:.1f} h (we will run 3–4 in parallel)")

Inventory loaded from store (locked).
Cases: 2238 | missing image: 0 | missing label: 0
Both manual+automatic label: 482 manual
Total image size: 194.4 GB | median 68.6 MB | max 889.7 MB
label      PDAC  non-PDAC
tier                     
automatic   197      1559
manual      479         3

Speed test: 100962_00001 (69 MB) downloaded in 1.8s → 38.1 MB/s | read in 0.8s | size (512, 512, 272) spacing (0.953, 0.953, 1.5)
Rough single-stream estimate for all 194 GB: 1.4 h (we will run 3–4 in parallel)


In [6]:
# ============================================================
# CELL 6 — LOCKED PATIENT-LEVEL SPLIT (70 / 15 / 15)
# Created once, stored in the repo, reused forever.
# ============================================================
from sklearn.model_selection import train_test_split

SPLIT_PATH = "meta/split.csv"
inv = inventory.copy()
inv["is_manual"] = inv["tier"].eq("manual")

if STORE.exists(SPLIT_PATH):
    split_df = pd.read_csv(STORE.get(SPLIT_PATH, force=True), dtype=str)
    print("Split loaded from store (LOCKED).")
else:
    pat = (inv.groupby("patient_id")
              .agg(label=("label", lambda s: "PDAC" if (s == "PDAC").any() else "non-PDAC"),
                   n_label=("label", "nunique"),
                   manual=("is_manual", "any"))
              .reset_index())
    print(f"Patients: {len(pat)} | patients with conflicting study labels: {(pat.n_label > 1).sum()}")

    def strata(df, min_n=10):
        s = df["label"] + "|" + df["manual"].map({True: "manual", False: "auto"})
        counts = s.value_counts()
        rare = s.isin(counts[counts < min_n].index)
        s[rare] = df.loc[rare, "label"] + "|auto"      # tiny groups merged into their label group
        return s

    train_p, rest_p = train_test_split(pat, test_size=0.30, stratify=strata(pat), random_state=SEED)
    val_p, test_p = train_test_split(rest_p, test_size=0.50, stratify=strata(rest_p), random_state=SEED)
    assign = {**{p: "train" for p in train_p.patient_id},
              **{p: "val" for p in val_p.patient_id},
              **{p: "test" for p in test_p.patient_id}}
    split_df = inv[["study_id", "patient_id"]].assign(split=inv.patient_id.map(assign))
    assert split_df.split.notna().all()
    split_df.to_csv(WORK / "split.csv", index=False)
    STORE.put(WORK / "split.csv", SPLIT_PATH, msg="LOCKED patient-level split")
    print("Split created and LOCKED in store.")

inv = inv.drop(columns=["split"], errors="ignore").merge(split_df[["study_id", "split"]], on="study_id")

# ---- checks ----
by_split = inv.groupby("split").patient_id.apply(set)
assert not (by_split["train"] & by_split["val"]) and not (by_split["train"] & by_split["test"]) \
       and not (by_split["val"] & by_split["test"]), "Patient leakage!"
print("✓ No patient appears in two splits")
print("\nPatients per split:", inv.groupby("split").patient_id.nunique().to_dict())
print("\nStudies: split × label")
print(pd.crosstab(inv.split, inv.label, margins=True))
print("\nStudies: split × tier")
print(pd.crosstab(inv.split, inv.tier, margins=True))
print("\nManual (gold) studies: split × label")
print(pd.crosstab(inv[inv.is_manual].split, inv[inv.is_manual].label))

Split loaded from store (LOCKED).
✓ No patient appears in two splits

Patients per split: {'test': 334, 'train': 1556, 'val': 334}

Studies: split × label
label  PDAC  non-PDAC   All
split                      
test    101       237   338
train   472      1091  1563
val     103       234   337
All     676      1562  2238

Studies: split × tier
tier   automatic  manual   All
split                         
test         266      72   338
train       1227     336  1563
val          263      74   337
All         1756     482  2238

Manual (gold) studies: split × label
label  PDAC  non-PDAC
split                
test     72         0
train   334         2
val      73         1


In [7]:
# ============================================================
# CELL 7 — SHARED CT PREPROCESSING + RESUMABLE STREAMING ENGINE
# Used by Pass A (localizer data), Pass B (all-case ROI crops) and test inference.
# ============================================================
import gc, tarfile
import numpy as np
import SimpleITK as sitk
import psutil
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from huggingface_hub import CommitOperationAdd

TARGET_SPACING_XYZ = (1.0, 1.0, 2.0)
FINE_SPACING_ZYX = np.array([2.0, 1.0, 1.0], dtype=np.float32)
HU_MIN, HU_MAX = -100.0, 400.0
BODY_THRESHOLD_HU = -500.0
BODY_MARGIN_MM = 16.0
COARSE_FACTOR = np.array([2, 4, 4])                    # fine (2,1,1 mm) → coarse (4,4,4 mm)
COARSE_SPACING_ZYX = FINE_SPACING_ZYX * COARSE_FACTOR
PANCREAS_LABELS = (1, 4, 5)                            # PDAC + parenchyma + duct
N_WORKERS = max(1, min(3, (os.cpu_count() or 2) - 1))
MAX_SESSION_HOURS = {"kaggle": 11.5, "colab": 11.0}.get(PLATFORM, 1e9)
KERNEL_START = psutil.Process(os.getpid()).create_time()
try:
    sitk.ProcessObject.SetGlobalDefaultNumberOfThreads(os.cpu_count() or 2)
except Exception:
    pass

def hours_used():
    return (time.time() - KERNEL_START) / 3600

def time_ok(margin_h=0.4):
    return hours_used() < MAX_SESSION_HOURS - margin_h

# ---------------- CT helpers ----------------
def download_case(row, with_label=True):
    d = SCRATCH / "raw" / row["study_id"]
    d.mkdir(parents=True, exist_ok=True)
    for attempt in range(4):
        try:
            ct = hf_hub_download(SOURCE_REPO, row["image_path"], repo_type="dataset", local_dir=str(d))
            lb = hf_hub_download(SOURCE_REPO, row["label_path"], repo_type="dataset",
                                 local_dir=str(d)) if with_label else None
            return Path(ct), (Path(lb) if lb else None), d
        except Exception:
            if attempt == 3:
                shutil.rmtree(d, ignore_errors=True)
                raise
            time.sleep(10 * (attempt + 1))

def load_resampled(ct_path, mask_path=None):
    """CT → RAS, 1×1×2 mm (X,Y,Z), float32 HU array [Z,Y,X]; mask nearest-neighbour on the same grid."""
    ct = sitk.DICOMOrient(sitk.ReadImage(str(ct_path)), "RAS")
    size, sp = ct.GetSize(), ct.GetSpacing()
    new_size = [max(1, int(round(size[i] * sp[i] / TARGET_SPACING_XYZ[i]))) for i in range(3)]
    rs = sitk.ResampleImageFilter()
    rs.SetOutputSpacing(TARGET_SPACING_XYZ)
    rs.SetSize(new_size)
    rs.SetOutputDirection(ct.GetDirection())
    rs.SetOutputOrigin(ct.GetOrigin())
    rs.SetInterpolator(sitk.sitkLinear)
    rs.SetDefaultPixelValue(-1000.0)
    rs.SetOutputPixelType(sitk.sitkFloat32)
    ct_r = rs.Execute(ct)
    del ct
    hu = sitk.GetArrayFromImage(ct_r)
    meta = {"orig_size": list(size), "orig_spacing": [round(s, 4) for s in sp], "fine_shape": list(hu.shape)}
    mask = None
    if mask_path is not None:
        m = sitk.DICOMOrient(sitk.ReadImage(str(mask_path), sitk.sitkUInt8), "RAS")
        orig_fg = int(np.count_nonzero(sitk.GetArrayViewFromImage(m)))
        orig_vox = float(np.prod(m.GetSpacing()))
        m_r = sitk.Resample(m, ct_r, sitk.Transform(), sitk.sitkNearestNeighbor, 0, sitk.sitkUInt8)
        del m
        mask = sitk.GetArrayFromImage(m_r)
        meta["mask_volume_ratio"] = round(np.count_nonzero(mask) * 2.0 / max(orig_fg * orig_vox, 1e-6), 3)
        meta["labels"] = [int(v) for v in np.unique(mask)]
        if mask.max() > 6:
            raise ValueError(f"unexpected label values {meta['labels']}")
    return hu, mask, meta

def body_bounds(hu):
    """CT-only body box with 16 mm margin. Returns inclusive (lo, hi) in [Z,Y,X]."""
    fg = hu > BODY_THRESHOLD_HU
    if not fg.any():
        raise ValueError("no body voxels")
    zz = np.flatnonzero(fg.any(axis=(1, 2)))
    yy = np.flatnonzero(fg.any(axis=(0, 2)))
    xx = np.flatnonzero(fg.any(axis=(0, 1)))
    m = np.ceil(BODY_MARGIN_MM / FINE_SPACING_ZYX).astype(int)
    lo = np.maximum(np.array([zz[0], yy[0], xx[0]]) - m, 0)
    hi = np.minimum(np.array([zz[-1], yy[-1], xx[-1]]) + m, np.array(hu.shape) - 1)
    return lo, hi

def to_unit(hu):
    return np.clip((hu - HU_MIN) / (HU_MAX - HU_MIN), 0.0, 1.0).astype(np.float32)

def coarse_downsample(unit):
    f = COARSE_FACTOR
    pad = [(0, (-s) % k) for s, k in zip(unit.shape, f)]
    u = np.pad(unit, pad, constant_values=0.0)
    z, y, x = u.shape
    return u.reshape(z // f[0], f[0], y // f[1], f[1], x // f[2], f[2]).mean(axis=(1, 3, 5)).astype(np.float32)

def fine_to_coarse(c):
    return (np.asarray(c, dtype=np.float64) - (COARSE_FACTOR - 1) / 2) / COARSE_FACTOR

def coarse_to_fine(c):
    return np.asarray(c, dtype=np.float64) * COARSE_FACTOR + (COARSE_FACTOR - 1) / 2

# ---------------- store helpers ----------------
def commit_files(pairs, msg, retries=6):
    for a in range(retries):
        try:
            ops = [CommitOperationAdd(path_in_repo=r, path_or_fileobj=str(l)) for l, r in pairs]
            api.create_commit(repo_id=WORK_REPO, repo_type=WORK_REPO_TYPE,
                              operations=ops, commit_message=msg)
            return
        except Exception as e:
            if a == retries - 1:
                raise
            print(f"   commit retry {a + 1}: {str(e)[:150]}")
            time.sleep(15 * (a + 1))

def run_stream_phase(phase, cohort, worker, out_dir, shard_size=50, max_failures=2):
    """worker(row, out_dir) -> (record_dict, [files]). Uploads a shard every `shard_size` cases.
    Re-running resumes: finished cases are skipped; cases failing `max_failures` times are excluded."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    idx_path = f"{phase}/index.json"
    index = STORE.get_json(idx_path) or {"done": {}, "shards": [], "failures": {}}
    excluded = {s for s, f in index["failures"].items() if f.get("count", 0) >= max_failures}
    pending = [r for _, r in cohort.iterrows()
               if r["study_id"] not in index["done"] and r["study_id"] not in excluded]
    print(f"[{phase}] cohort {len(cohort)} | done {len(index['done'])} | excluded {len(excluded)} | "
          f"pending {len(pending)} | workers {N_WORKERS} | session hours used {hours_used():.2f}")
    state = {"buffer": [], "ok": 0, "fail": 0, "stopped": False}
    t0 = time.time()

    def save_index():
        p = SCRATCH / f"{phase}_index.json"
        p.write_text(json.dumps(index, default=str))
        return p

    def flush():
        buf = state["buffer"]
        if not buf:
            return
        name = f"shard_{len(index['shards']):04d}"
        tar_path = SCRATCH / f"{phase}_{name}.tar"
        with tarfile.open(tar_path, "w") as tar:
            for _, files in buf:
                for f in files:
                    tar.add(f, arcname=Path(f).name)
        for rec, _ in buf:
            index["done"][rec["study_id"]] = {**rec, "shard": name}
            index["failures"].pop(rec["study_id"], None)
        index["shards"].append(name)
        commit_files([(tar_path, f"{phase}/shards/{name}.tar"), (save_index(), idx_path)],
                     msg=f"{phase}: {name} (+{len(buf)}) total {len(index['done'])}")
        tar_path.unlink(missing_ok=True)
        (out_dir / f".{name}.ok").touch()
        state["buffer"] = []
        print(f"   ↑ saved {name} | total done {len(index['done'])}")

    it = iter(pending)
    inflight = {}
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        def submit_next():
            if state["stopped"] or not time_ok():
                state["stopped"] = state["stopped"] or not time_ok()
                return
            r = next(it, None)
            if r is not None:
                inflight[ex.submit(worker, r, out_dir)] = r

        for _ in range(N_WORKERS * 2):
            submit_next()
        while inflight:
            finished, _ = wait(list(inflight), return_when=FIRST_COMPLETED)
            for fut in finished:
                r = inflight.pop(fut)
                sid = r["study_id"]
                try:
                    state["buffer"].append(fut.result())
                    state["ok"] += 1
                except Exception as e:
                    state["fail"] += 1
                    prev = index["failures"].get(sid, {}).get("count", 0)
                    index["failures"][sid] = {"error": f"{type(e).__name__}: {e}"[:300], "count": prev + 1}
                    print(f"   ✗ {sid}: {index['failures'][sid]['error'][:160]}")
                n = state["ok"] + state["fail"]
                if n % 25 == 0:
                    rate = n / (time.time() - t0)
                    eta = (len(pending) - n) / max(rate, 1e-9) / 3600
                    print(f"   {n}/{len(pending)} | ok {state['ok']} fail {state['fail']} | "
                          f"{rate * 60:.1f} cases/min | ETA {eta:.2f} h | RAM {psutil.virtual_memory().percent:.0f}%")
                submit_next()
            if len(state["buffer"]) >= shard_size:
                flush()
            gc.collect()
    flush()
    if state["fail"]:
        commit_files([(save_index(), idx_path)], msg=f"{phase}: failures update")
    if state["stopped"]:
        print(f"⏸ [{phase}] stopped before the session time limit. Re-run to resume.")
    print(f"[{phase}] this run: ok {state['ok']} | failed {state['fail']} | {(time.time() - t0) / 60:.1f} min")
    return index

def fetch_phase(phase, dest):
    """Download + extract every stored shard of a phase (skips shards already present)."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    index = STORE.get_json(f"{phase}/index.json")
    if not index:
        raise RuntimeError(f"No stored data for phase '{phase}'.")
    for name in index["shards"]:
        if (dest / f".{name}.ok").exists():
            continue
        p = Path(hf_hub_download(WORK_REPO, f"{phase}/shards/{name}.tar", repo_type=WORK_REPO_TYPE,
                                 local_dir=str(SCRATCH / "dl"), token=HF_TOKEN))
        with tarfile.open(p) as tar:
            tar.extractall(dest, filter="data")
        p.unlink(missing_ok=True)
        (dest / f".{name}.ok").touch()
    return index

print(f"✓ Helpers ready | workers {N_WORKERS} | session limit {MAX_SESSION_HOURS} h | "
      f"hours used so far {hours_used():.2f}")

✓ Helpers ready | workers 3 | session limit 11.5 h | hours used so far 0.01


In [8]:
# ============================================================
# CELL 8 — PASS A: LOCALIZER DATA (TRAIN + VAL only, CT-only inputs)
# Saves a 4 mm coarse body CT + true pancreas centre per case.
# Resumable: re-run after any interruption.
# ============================================================
PHASE_A = "passA"
PASSA_N_AUTO_TRAIN = 500
PASSA_N_AUTO_VAL = 120
PASSA_DIR = SCRATCH / "passA_cache"

COHORT_A_PATH = f"{PHASE_A}/cohort.csv"
if STORE.exists(COHORT_A_PATH):
    cohort_A = pd.read_csv(STORE.get(COHORT_A_PATH, force=True), dtype=str)
    print("Pass A cohort loaded (LOCKED).")
else:
    def pick_auto(split, n):
        a = inv[(inv.split == split) & (inv.tier == "automatic")]
        pdac, healthy = a[a.label == "PDAC"], a[a.label != "PDAC"]
        n_pdac = min(len(pdac), n // 4)
        return pd.concat([pdac.sample(n_pdac, random_state=SEED),
                          healthy.sample(n - n_pdac, random_state=SEED)])
    cohort_A = pd.concat([
        inv[(inv.split.isin(["train", "val"])) & (inv.tier == "manual")],
        pick_auto("train", PASSA_N_AUTO_TRAIN),
        pick_auto("val", PASSA_N_AUTO_VAL),
    ])[["study_id", "patient_id", "split", "tier", "label", "image_path", "label_path", "image_mb"]]
    assert not (cohort_A.split == "test").any()
    cohort_A.to_csv(WORK / "passA_cohort.csv", index=False)
    STORE.put(WORK / "passA_cohort.csv", COHORT_A_PATH, msg="LOCKED Pass A cohort")
    print("Pass A cohort created and LOCKED.")

print(pd.crosstab([cohort_A.split, cohort_A.tier], cohort_A.label, margins=True))
print(f"Download volume: {cohort_A.image_mb.astype(float).sum() / 1e3:.1f} GB")


def passA_worker(row, out_dir):
    t0 = time.time()
    sid = row["study_id"]
    ct_p, lb_p, d = download_case(row)
    try:
        hu, mask, meta = load_resampled(ct_p, lb_p)
    finally:
        shutil.rmtree(d, ignore_errors=True)
    tumor_total = int(np.count_nonzero(mask == 1))
    lo, hi = body_bounds(hu)
    sl = tuple(slice(a, b + 1) for a, b in zip(lo, hi))
    hu_b, m_b = hu[sl], mask[sl]
    del hu, mask
    panc = np.isin(m_b, PANCREAS_LABELS)
    n_panc = int(panc.sum())
    if n_panc == 0:
        raise ValueError("no pancreas/PDAC voxels in mask")
    coords = np.argwhere(panc)
    c_fine = coords.mean(axis=0)
    ext_mm = ((coords.max(axis=0) - coords.min(axis=0) + 1) * FINE_SPACING_ZYX).round(1)
    n_tumor = int(np.count_nonzero(m_b == 1))
    coarse = coarse_downsample(to_unit(hu_b))
    c_coarse = fine_to_coarse(c_fine)
    f = Path(out_dir) / f"{sid}.npz"
    np.savez_compressed(f, image=coarse.astype(np.float16), center=c_coarse.astype(np.float32))
    rec = {
        "study_id": sid, "coarse_shape": list(coarse.shape), "body_shape": list(hu_b.shape),
        "body_lo": lo.tolist(), "center_fine": [round(float(v), 2) for v in c_fine],
        "pancreas_extent_mm": ext_mm.tolist(), "n_pancreas_vox": n_panc, "n_tumor_vox": n_tumor,
        "tumor_outside_body": tumor_total - n_tumor, "mask_volume_ratio": meta["mask_volume_ratio"],
        "labels": meta["labels"], "orig_spacing": meta["orig_spacing"], "seconds": round(time.time() - t0, 1),
    }
    del hu_b, m_b, panc, coords, coarse
    return rec, [f]


if STORE.done(PHASE_A):
    index_A = STORE.get_json(f"{PHASE_A}/index.json")
    print(f"✓ Pass A already complete ({len(index_A['done'])} cases) — skipping.")
else:
    index_A = run_stream_phase(PHASE_A, cohort_A, passA_worker, PASSA_DIR)
    remaining = set(cohort_A.study_id) - set(index_A["done"])
    excluded = {s for s, f in index_A["failures"].items() if f.get("count", 0) >= 2}
    if not (remaining - excluded):
        STORE.mark_done(PHASE_A, {"n_done": len(index_A["done"]), "excluded": sorted(excluded)})
        print("✓ Pass A COMPLETE and marked done.")
    else:
        print(f"Pass A not finished: {len(remaining - excluded)} cases left. Re-run this cell.")

# ---- summary ----
recs = pd.DataFrame(index_A["done"].values())
if len(recs):
    recs = recs.merge(cohort_A[["study_id", "split", "tier", "label"]], on="study_id", how="left")
    print("\nDone by split × tier × label:")
    print(pd.crosstab([recs.split, recs.tier], recs.label))
    print(f"\nSeconds/case: median {recs.seconds.median():.1f} | p95 {recs.seconds.quantile(.95):.1f}")
    print(f"Mask volume ratio after resampling: median {recs.mask_volume_ratio.median():.3f} | "
          f"outside [0.8,1.25]: {((recs.mask_volume_ratio < 0.8) | (recs.mask_volume_ratio > 1.25)).sum()}")
    print(f"Cases with tumour voxels outside body crop: {(recs.tumor_outside_body > 0).sum()}")
    ext = np.array(recs.pancreas_extent_mm.tolist())
    print(f"Pancreas extent mm [Z,Y,X]: median {np.median(ext, 0).round(0)} | max {ext.max(0).round(0)}")
if index_A["failures"]:
    print("\nFailures:", {k: v["error"][:80] for k, v in list(index_A["failures"].items())[:10]})

Pass A cohort loaded (LOCKED).
label            PDAC  non-PDAC   All
split tier                           
train automatic   125       375   500
      manual      334         2   336
val   automatic    30        90   120
      manual       73         1    74
All               562       468  1030
Download volume: 93.2 GB
✓ Pass A already complete (1030 cases) — skipping.

Done by split × tier × label:
label            PDAC  non-PDAC
split tier                     
train automatic   125       375
      manual      334         2
val   automatic    30        90
      manual       73         1

Seconds/case: median 7.8 | p95 16.9
Mask volume ratio after resampling: median 1.000 | outside [0.8,1.25]: 0
Cases with tumour voxels outside body crop: 0
Pancreas extent mm [Z,Y,X]: median [ 84.  73. 122.] | max [176. 133. 228.]


In [9]:
# ============================================================
# CELL 9 — STAGE 1: PANCREAS LOCALIZER (CPU or GPU, resumable) + PRE-REGISTERED GATE
# Checkpoints go to the repo every 2 epochs; re-running resumes from the last one.
# ============================================================
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

PHASE_L = "localizer"
L_EPOCHS, L_LR, L_WD, L_HUBER, L_VAL_EVERY = 40, 2e-4, 1e-4, 0.10, 2
B1_SHAPE = np.array([96, 224, 288])                    # Stage-2 ROI, fine voxels [Z,Y,X]
B1_MM = B1_SHAPE * FINE_SPACING_ZYX                    # 192 × 224 × 288 mm
DEV_L = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 2)

# ---- PRE-REGISTERED GATE (written before seeing results) ----
GATE = {
    "manual_val":    {"median_mm_max": 25.0, "within40_min": 0.90},
    "automatic_val": {"median_mm_max": 30.0, "within40_min": 0.85},
    "roi_containment_min": 0.95,
}

# ---- data ----
index_A = fetch_phase("passA", PASSA_DIR)
meta_A = (pd.DataFrame(index_A["done"].values())
            .merge(cohort_A[["study_id", "split", "tier", "label"]], on="study_id"))
DATA = {}
for sid in meta_A.study_id:
    with np.load(PASSA_DIR / f"{sid}.npz") as d:
        DATA[sid] = (d["image"].astype(np.float32), d["center"].astype(np.float32))
train_ids = meta_A[meta_A.split == "train"].study_id.tolist()
val_ids = meta_A[meta_A.split == "val"].study_id.tolist()
print(f"Localizer data: train {len(train_ids)} | val {len(val_ids)} | device {DEV_L}")

def coord_channels(img):
    z, y, x = img.shape
    zc = np.broadcast_to(np.linspace(-1, 1, z, dtype=np.float32)[:, None, None], img.shape)
    yc = np.broadcast_to(np.linspace(-1, 1, y, dtype=np.float32)[None, :, None], img.shape)
    xc = np.broadcast_to(np.linspace(-1, 1, x, dtype=np.float32)[None, None, :], img.shape)
    return np.stack([img, zc, yc, xc]).astype(np.float32)

def norm_target(c, shape):
    return (2.0 * np.asarray(c) / np.maximum(np.asarray(shape) - 1.0, 1.0) - 1.0).astype(np.float32)

def denorm_target(t, shape):
    return (np.asarray(t) + 1.0) * 0.5 * (np.asarray(shape) - 1.0)

def augment(img, rng):
    img = img.copy()
    if rng.random() < 0.5: img *= rng.uniform(0.90, 1.10)
    if rng.random() < 0.5: img += rng.uniform(-0.05, 0.05)
    if rng.random() < 0.3: img += rng.normal(0, 0.015, img.shape).astype(np.float32)
    return np.clip(img, 0, 1).astype(np.float32)

class Localizer(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o):
            return [nn.Conv3d(i, o, 3, padding=1, bias=False), nn.InstanceNorm3d(o), nn.ReLU(inplace=True)]
        self.features = nn.Sequential(*block(4, 16), nn.MaxPool3d(2), *block(16, 32), nn.MaxPool3d(2),
                                      *block(32, 64), nn.MaxPool3d(2), *block(64, 128), nn.AdaptiveAvgPool3d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(128, 64), nn.ReLU(inplace=True),
                                  nn.Dropout(0.10), nn.Linear(64, 3), nn.Tanh())
    def forward(self, x):
        return self.head(self.features(x))

def predict_center_coarse(model, img):
    model.eval()
    with torch.inference_mode():
        t = model(torch.from_numpy(coord_channels(img))[None].to(DEV_L))[0].float().cpu().numpy()
    return denorm_target(t, img.shape)

def evaluate(model, ids):
    rows = []
    for sid in ids:
        img, c = DATA[sid]
        err_axes_mm = (predict_center_coarse(model, img) - c) * COARSE_SPACING_ZYX
        rows.append({"study_id": sid, "err_mm": float(np.linalg.norm(err_axes_mm)),
                     "ez": err_axes_mm[0], "ey": err_axes_mm[1], "ex": err_axes_mm[2]})
    return pd.DataFrame(rows)

# ---- resume ----
model = Localizer().to(DEV_L)
opt = torch.optim.AdamW(model.parameters(), lr=L_LR, weight_decay=L_WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=L_EPOCHS, eta_min=1e-6)
LDIR = WORK / "localizer"; LDIR.mkdir(exist_ok=True)
start_epoch, best_err, history = 0, float("inf"), []
if STORE.exists(f"{PHASE_L}/last.pt"):
    ck = torch.load(STORE.get(f"{PHASE_L}/last.pt", force=True), map_location=DEV_L, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start_epoch, best_err, history = ck["epoch"] + 1, ck["best_err"], ck["history"]
    print(f"Resumed from epoch {ck['epoch'] + 1} | best val mean error {best_err:.1f} mm")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

def save_and_upload(epoch, improved):
    ck = {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
          "epoch": epoch, "best_err": best_err, "history": history}
    torch.save(ck, LDIR / "last.pt")
    pd.DataFrame(history).to_csv(LDIR / "history.csv", index=False)
    pairs = [(LDIR / "last.pt", f"{PHASE_L}/last.pt"), (LDIR / "history.csv", f"{PHASE_L}/history.csv")]
    if improved:
        pairs.append((LDIR / "best.pt", f"{PHASE_L}/best.pt"))
    commit_files(pairs, msg=f"localizer epoch {epoch + 1}")

if not STORE.done(PHASE_L):
    for epoch in range(start_epoch, L_EPOCHS):
        t0 = time.time()
        model.train()
        rng = np.random.default_rng(SEED + epoch)
        losses = []
        for sid in rng.permutation(train_ids):
            img, c = DATA[sid]
            x = torch.from_numpy(coord_channels(augment(img, rng)))[None].to(DEV_L)
            t = torch.from_numpy(norm_target(c, img.shape))[None].to(DEV_L)
            loss = F.smooth_l1_loss(model(x), t, beta=L_HUBER)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            losses.append(loss.item())
        sched.step()
        rec = {"epoch": epoch + 1, "train_loss": float(np.mean(losses)), "lr": opt.param_groups[0]["lr"],
               "sec": round(time.time() - t0, 1)}
        improved = False
        if (epoch + 1) % L_VAL_EVERY == 0 or epoch + 1 == L_EPOCHS:
            ev = evaluate(model, val_ids)
            rec.update({"val_mean_mm": ev.err_mm.mean(), "val_median_mm": ev.err_mm.median(),
                        "val_within40": (ev.err_mm <= 40).mean()})
            if rec["val_mean_mm"] < best_err:
                best_err, improved = rec["val_mean_mm"], True
                torch.save({"model": model.state_dict(), "epoch": epoch + 1, "val_mean_mm": best_err}, LDIR / "best.pt")
        history.append(rec)
        msg = (f"Epoch {epoch + 1:03d} | loss {rec['train_loss']:.5f} | {rec['sec']:.0f}s")
        if "val_mean_mm" in rec:
            msg += (f" | val mean {rec['val_mean_mm']:.1f} mm median {rec['val_median_mm']:.1f} mm "
                    f"≤40mm {rec['val_within40']:.1%}" + (" ★" if improved else ""))
        print(msg)
        if "val_mean_mm" in rec:
            save_and_upload(epoch, improved)
        if not time_ok():
            save_and_upload(epoch, False)
            print("⏸ Session time nearly used up — checkpoint saved. Re-run to resume.")
            break

# ---- gate on the BEST checkpoint ----
if len(history) and history[-1]["epoch"] == L_EPOCHS or STORE.done(PHASE_L):
    best = torch.load(STORE.get(f"{PHASE_L}/best.pt", force=True), map_location=DEV_L, weights_only=False)
    model.load_state_dict(best["model"])
    ev = evaluate(model, val_ids).merge(meta_A[["study_id", "tier", "label", "pancreas_extent_mm"]], on="study_id")
    ext = np.array(ev.pancreas_extent_mm.tolist())
    err = np.abs(ev[["ez", "ey", "ex"]].to_numpy())
    ev["roi_contains_pancreas_approx"] = ((err + ext / 2) <= (B1_MM / 2)).all(axis=1)

    def summ(d):
        return {"n": len(d), "mean_mm": round(d.err_mm.mean(), 1), "median_mm": round(d.err_mm.median(), 1),
                "within20": round((d.err_mm <= 20).mean(), 3), "within40": round((d.err_mm <= 40).mean(), 3),
                "roi_contain": round(d.roi_contains_pancreas_approx.mean(), 3)}
    res = {"best_epoch": best["epoch"],
           "manual_val": summ(ev[ev.tier == "manual"]),
           "automatic_val": summ(ev[ev.tier == "automatic"]),
           "automatic_val_nonPDAC": summ(ev[(ev.tier == "automatic") & (ev.label != "PDAC")]),
           "automatic_val_PDAC": summ(ev[(ev.tier == "automatic") & (ev.label == "PDAC")]),
           "all_val": summ(ev)}
    for k, v in res.items():
        print(f"{k:24s}: {v}")
    checks = {
        "manual median": res["manual_val"]["median_mm"] <= GATE["manual_val"]["median_mm_max"],
        "manual ≤40mm": res["manual_val"]["within40"] >= GATE["manual_val"]["within40_min"],
        "automatic median": res["automatic_val"]["median_mm"] <= GATE["automatic_val"]["median_mm_max"],
        "automatic ≤40mm": res["automatic_val"]["within40"] >= GATE["automatic_val"]["within40_min"],
        "ROI containment": res["all_val"]["roi_contain"] >= GATE["roi_containment_min"],
    }
    print("\nPRE-REGISTERED GATE:")
    for k, ok in checks.items():
        print(f"   {'✓' if ok else '✗'} {k}")
    ev.to_csv(LDIR / "val_errors.csv", index=False)
    commit_files([(LDIR / "val_errors.csv", f"{PHASE_L}/val_errors.csv")], msg="localizer val errors")
    if all(checks.values()):
        if not STORE.done(PHASE_L):
            STORE.mark_done(PHASE_L, res)
        print("\n✓ GATE PASSED — localizer locked. Next: Pass B.")
    else:
        print("\n✗ GATE FAILED — do NOT continue. Review the output above.")
        print("Worst 10 val cases:")
        print(ev.sort_values("err_mm", ascending=False).head(10)[["study_id", "tier", "label", "err_mm", "ez", "ey", "ex"]])

Localizer data: train 836 | val 194 | device cuda
Resumed from epoch 40 | best val mean error 18.7 mm
Parameters: 300,483
best_epoch              : 38
manual_val              : {'n': 74, 'mean_mm': np.float64(18.5), 'median_mm': 15.1, 'within20': np.float64(0.608), 'within40': np.float64(0.973), 'roi_contain': np.float64(1.0)}
automatic_val           : {'n': 120, 'mean_mm': np.float64(18.9), 'median_mm': 17.9, 'within20': np.float64(0.575), 'within40': np.float64(0.983), 'roi_contain': np.float64(1.0)}
automatic_val_nonPDAC   : {'n': 90, 'mean_mm': np.float64(18.6), 'median_mm': 17.2, 'within20': np.float64(0.6), 'within40': np.float64(0.978), 'roi_contain': np.float64(1.0)}
automatic_val_PDAC      : {'n': 30, 'mean_mm': np.float64(19.7), 'median_mm': 20.7, 'within20': np.float64(0.5), 'within40': np.float64(1.0), 'roi_contain': np.float64(1.0)}
all_val                 : {'n': 194, 'mean_mm': np.float64(18.7), 'median_mm': 17.1, 'within20': np.float64(0.588), 'within40': np.float64(0.9

No files have been modified since last commit. Skipping to prevent empty commit.



✓ GATE PASSED — localizer locked. Next: Pass B.


In [10]:
# ============================================================
# CELL 10 — PASS B: CT-ONLY ROI CROPS FOR ALL 2,238 CASES (nnU-Net raw format)
# Crop centre comes ONLY from the locked localizer (CT input). Masks are cropped with the
# same box and used for training targets (train/val) and evaluation/audit only (test).
# Resumable: re-run after any interruption.
# ============================================================
PHASE_B = "passB"
PASSB_DIR = SCRATCH / "passB_cache"
if not STORE.done(PHASE_L):
    raise RuntimeError("Localizer gate has not passed — Pass B is blocked. Check the cell 9 output.")

# ---- lock the localizer + ROI config used for every crop ----
loc_ck = torch.load(STORE.get(f"{PHASE_L}/best.pt", force=True), map_location="cpu", weights_only=False)
LOC_MODEL = Localizer()
LOC_MODEL.load_state_dict(loc_ck["model"])
LOC_MODEL.eval()
DEV_L = torch.device("cpu")
LOC_MODEL.to(DEV_L)
B_CONFIG = {"localizer_best_epoch": int(loc_ck["epoch"]), "roi_shape_zyx": B1_SHAPE.tolist(),
            "spacing_xyz": list(TARGET_SPACING_XYZ), "hu_store_range": [-1024, 1500],
            "targets": {"0": "background", "1": "pancreas (labels 4,5)", "2": "PDAC (label 1)"}}
stored_cfg = STORE.get_json(f"{PHASE_B}/config.json")
if stored_cfg is None:
    STORE.put_json(B_CONFIG, f"{PHASE_B}/config.json", msg="LOCKED Pass B config")
elif stored_cfg != B_CONFIG:
    raise RuntimeError(f"Pass B config changed since the pass started:\nstored {stored_cfg}\nnow    {B_CONFIG}")
print("Pass B config:", B_CONFIG)

order = {"train": 0, "val": 1, "test": 2}
cohort_B = (inv.assign(_o=inv.split.map(order)).sort_values(["_o", "study_id"])
               [["study_id", "patient_id", "split", "tier", "label", "image_path", "label_path", "image_mb"]])
print(pd.crosstab(cohort_B.split, cohort_B.label, margins=True))

def crop_pad(arr, start, shape, fill):
    out = np.full(tuple(shape), fill, dtype=arr.dtype)
    s = np.asarray(start)
    e = s + np.asarray(shape)
    lo = np.maximum(s, 0)
    hi = np.minimum(e, arr.shape)
    if np.any(hi <= lo):
        return out
    d_lo = lo - s
    d_hi = d_lo + (hi - lo)
    out[tuple(slice(a, b) for a, b in zip(d_lo, d_hi))] = arr[tuple(slice(a, b) for a, b in zip(lo, hi))]
    return out

def passB_worker(row, out_dir):
    t0 = time.time()
    sid = row["study_id"]
    ct_p, lb_p, d = download_case(row)
    try:
        hu, mask, meta = load_resampled(ct_p, lb_p)
    finally:
        shutil.rmtree(d, ignore_errors=True)
    lo, hi = body_bounds(hu)
    sl = tuple(slice(a, b + 1) for a, b in zip(lo, hi))
    coarse = coarse_downsample(to_unit(hu[sl]))
    c_coarse = predict_center_coarse(LOC_MODEL, coarse)            # CT-only prediction
    c_fine = coarse_to_fine(c_coarse) + lo                          # full fine grid [Z,Y,X]
    start = np.round(c_fine - B1_SHAPE / 2).astype(int)
    roi_hu = crop_pad(hu, start, B1_SHAPE, np.float32(-1000.0))
    roi_m = crop_pad(mask, start, B1_SHAPE, np.uint8(0))
    # audit (ground truth used ONLY to measure what the CT-only crop kept)
    panc_all = int(np.isin(mask, PANCREAS_LABELS).sum())
    panc_in = int(np.isin(roi_m, PANCREAS_LABELS).sum())
    tum_all = int((mask == 1).sum())
    tum_in = int((roi_m == 1).sum())
    del hu, mask
    tgt = np.zeros(roi_m.shape, dtype=np.uint8)
    tgt[np.isin(roi_m, (4, 5))] = 1
    tgt[roi_m == 1] = 2
    img = sitk.GetImageFromArray(np.clip(roi_hu, -1024, 1500).astype(np.int16))
    img.SetSpacing(TARGET_SPACING_XYZ)
    lab = sitk.GetImageFromArray(tgt)
    lab.SetSpacing(TARGET_SPACING_XYZ)
    fi = Path(out_dir) / f"{sid}_0000.nii.gz"
    fl = Path(out_dir) / f"{sid}.nii.gz"
    sitk.WriteImage(img, str(fi), True)
    sitk.WriteImage(lab, str(fl), True)
    rec = {
        "study_id": sid, "roi_start_fine": start.tolist(), "body_lo": lo.tolist(),
        "pred_center_fine": [round(float(v), 1) for v in c_fine], "fine_shape": meta["fine_shape"],
        "orig_size": meta["orig_size"], "orig_spacing": meta["orig_spacing"],
        "pancreas_vox": panc_all, "pancreas_in_roi": panc_in, "tumor_vox": tum_all, "tumor_in_roi": tum_in,
        "mask_volume_ratio": meta["mask_volume_ratio"], "mb": round((fi.stat().st_size + fl.stat().st_size) / 1e6, 2),
        "seconds": round(time.time() - t0, 1),
    }
    del roi_hu, roi_m, tgt, img, lab
    return rec, [fi, fl]

if STORE.done(PHASE_B):
    index_B = STORE.get_json(f"{PHASE_B}/index.json")
    print(f"✓ Pass B already complete ({len(index_B['done'])} cases) — skipping.")
else:
    index_B = run_stream_phase(PHASE_B, cohort_B, passB_worker, PASSB_DIR)
    remaining = set(cohort_B.study_id) - set(index_B["done"])
    excluded = {s for s, f in index_B["failures"].items() if f.get("count", 0) >= 2}
    if not (remaining - excluded):
        STORE.mark_done(PHASE_B, {"n_done": len(index_B["done"]), "excluded": sorted(excluded)})
        print("✓ Pass B COMPLETE and marked done.")
    else:
        print(f"Pass B not finished: {len(remaining - excluded)} cases left. Re-run (or Save & Run All again).")

# ---- summary ----
rb = pd.DataFrame(index_B["done"].values())
if len(rb):
    rb = rb.merge(cohort_B[["study_id", "split", "tier", "label"]], on="study_id", how="left")
    rb["panc_frac"] = rb.pancreas_in_roi / rb.pancreas_vox.clip(lower=1)
    rb["tum_frac"] = np.where(rb.tumor_vox > 0, rb.tumor_in_roi / rb.tumor_vox.clip(lower=1), np.nan)
    print(f"\nDone: {len(rb)} | stored size {rb.mb.sum() / 1e3:.1f} GB | sec/case median {rb.seconds.median():.1f}")
    for (sp, tier), g in rb.groupby(["split", "tier"]):
        t = g[g.tumor_vox > 0]
        print(f"{sp:5s} {tier:9s} n={len(g):4d} | pancreas ≥99% inside ROI: {(g.panc_frac >= 0.99).mean():.1%} | "
              f"tumour cases {len(t):3d}, tumour 100% inside: {(t.tum_frac >= 1.0).mean() if len(t) else float('nan'):.1%}, "
              f"≥95% inside: {(t.tum_frac >= 0.95).mean() if len(t) else float('nan'):.1%}")
    worst = rb[rb.tumor_vox > 0].sort_values("tum_frac").head(8)
    print("\nLowest tumour coverage cases:")
    print(worst[["study_id", "split", "tier", "tumor_vox", "tum_frac", "panc_frac"]].to_string(index=False))
if index_B["failures"]:
    print("\nFailures:", {k: v["error"][:80] for k, v in list(index_B["failures"].items())[:10]})

Pass B config: {'localizer_best_epoch': 38, 'roi_shape_zyx': [96, 224, 288], 'spacing_xyz': [1.0, 1.0, 2.0], 'hu_store_range': [-1024, 1500], 'targets': {'0': 'background', '1': 'pancreas (labels 4,5)', '2': 'PDAC (label 1)'}}
label  PDAC  non-PDAC   All
split                      
test    101       237   338
train   472      1091  1563
val     103       234   337
All     676      1562  2238
✓ Pass B already complete (2238 cases) — skipping.

Done: 2238 | stored size 17.0 GB | sec/case median 11.2
test  automatic n= 266 | pancreas ≥99% inside ROI: 100.0% | tumour cases  29, tumour 100% inside: 100.0%, ≥95% inside: 100.0%
test  manual    n=  72 | pancreas ≥99% inside ROI: 98.6% | tumour cases  72, tumour 100% inside: 100.0%, ≥95% inside: 100.0%
train automatic n=1227 | pancreas ≥99% inside ROI: 99.0% | tumour cases 138, tumour 100% inside: 99.3%, ≥95% inside: 99.3%
train manual    n= 336 | pancreas ≥99% inside ROI: 99.7% | tumour cases 334, tumour 100% inside: 99.4%, ≥95% inside: 99.7%


In [11]:
# ============================================================
# CELL 11 — nnU-Net DATASET + PLAN + PREPROCESS (run in a CPU session)
# First run: builds Dataset501, plans ResEnc-M 3d_fullres, preprocesses, uploads to the repo.
# Later runs (e.g. GPU sessions): just restores the preprocessed data from the repo.
# ============================================================
import subprocess

PHASE_N = "nnunet_prep"
DS_ID, DS_NAME = 501, "Dataset501_PANORAMA"
PLANS = "nnUNetResEncUNetMPlans"
NN_RAW, NN_PRE, NN_RES = SCRATCH / "nnUNet_raw", SCRATCH / "nnUNet_preprocessed", SCRATCH / "nnUNet_results"
for k, p in [("nnUNet_raw", NN_RAW), ("nnUNet_preprocessed", NN_PRE), ("nnUNet_results", NN_RES)]:
    p.mkdir(parents=True, exist_ok=True)
    os.environ[k] = str(p)
PRE_DS = NN_PRE / DS_NAME

if not STORE.done(PHASE_B):
    raise RuntimeError("Pass B is not complete.")

# ---- case table + training exclusions (CT-only crop missed the target) ----
index_B = STORE.get_json(f"{PHASE_B}/index.json")
rb = (pd.DataFrame(index_B["done"].values())
        .merge(cohort_B[["study_id", "split", "tier", "label"]], on="study_id"))
rb["panc_frac"] = rb.pancreas_in_roi / rb.pancreas_vox.clip(lower=1)
rb["tum_frac"] = np.where(rb.tumor_vox > 0, rb.tumor_in_roi / rb.tumor_vox.clip(lower=1), 1.0)
EXCLUDED_TRAIN = rb[(rb.split == "train") & ((rb.panc_frac < 0.5) | (rb.tum_frac < 0.5))].study_id.tolist()
TRAIN_KEYS = sorted(rb[(rb.split == "train") & ~rb.study_id.isin(EXCLUDED_TRAIN)].study_id)
VAL_KEYS = sorted(rb[rb.split == "val"].study_id)
TEST_KEYS = sorted(rb[rb.split == "test"].study_id)
print(f"Train {len(TRAIN_KEYS)} | Val {len(VAL_KEYS)} | Test {len(TEST_KEYS)} | "
      f"excluded from training (crop missed target): {EXCLUDED_TRAIN}")


def nn_cli(module, func, args, log_name, show=("Error", "error", "Traceback", "Done", "done", "fingerprint", "Preprocessing")):
    """Run an nnU-Net entry point in a subprocess; full log saved, key lines printed."""
    code = f"import sys; sys.argv = ['nnunet'] + {list(map(str, args))!r}; from {module} import {func}; {func}()"
    log_path = WORK / log_name
    print("▶", func, " ".join(map(str, args)))
    t0 = time.time()
    with open(log_path, "w") as log:
        proc = subprocess.Popen([sys.executable, "-c", code], stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
        for line in proc.stdout:
            log.write(line)
            if any(k in line for k in show) and "%|" not in line:
                print("   ", line.rstrip()[:220])
        rc = proc.wait()
    print(f"   finished in {(time.time() - t0) / 60:.1f} min (exit code {rc})")
    if rc != 0:
        print("".join(open(log_path).readlines()[-60:]))
        raise RuntimeError(f"{func} failed — see the lines above.")
    return log_path


def upload_preprocessed(part_gb=2.0):
    files = sorted(p for p in PRE_DS.rglob("*") if p.is_file())
    parts, cur, size = [], [], 0
    for f in files:
        cur.append(f)
        size += f.stat().st_size
        if size >= part_gb * 1e9:
            parts.append(cur); cur, size = [], 0
    if cur:
        parts.append(cur)
    manifest = {"parts": [f"part_{i:03d}" for i in range(len(parts))], "n_files": len(files)}
    for i, group in enumerate(parts):
        name = f"part_{i:03d}"
        if STORE.exists(f"{PHASE_N}/{name}.tar"):
            continue
        tar_path = SCRATCH / f"{name}.tar"
        with tarfile.open(tar_path, "w") as tar:
            for f in group:
                tar.add(f, arcname=str(f.relative_to(NN_PRE)))
        commit_files([(tar_path, f"{PHASE_N}/{name}.tar")], msg=f"nnU-Net preprocessed {name}")
        tar_path.unlink(missing_ok=True)
        print(f"   ↑ {name} ({len(group)} files)")
    STORE.put_json(manifest, f"{PHASE_N}/manifest.json", msg="nnU-Net preprocessed manifest")
    return manifest


def restore_preprocessed():
    manifest = STORE.get_json(f"{PHASE_N}/manifest.json")
    t0 = time.time()
    for name in manifest["parts"]:
        marker = NN_PRE / f".{name}.ok"
        if marker.exists():
            continue
        p = Path(hf_hub_download(WORK_REPO, f"{PHASE_N}/{name}.tar", repo_type=WORK_REPO_TYPE,
                                 local_dir=str(SCRATCH / "dl"), token=HF_TOKEN))
        with tarfile.open(p) as tar:
            tar.extractall(NN_PRE, filter="data")
        p.unlink(missing_ok=True)
        marker.touch()
    print(f"✓ Preprocessed data restored ({manifest['n_files']} files) in {(time.time() - t0) / 60:.1f} min")


def print_plans():
    plans = json.load(open(PRE_DS / f"{PLANS}.json"))
    c = plans["configurations"]["3d_fullres"]
    a = c["architecture"]
    summary = {"patch_size": c["patch_size"], "batch_size": c["batch_size"], "spacing": c["spacing"],
               "median_image_size": c.get("median_image_size_in_voxels"),
               "network": a["network_class_name"].split(".")[-1], "n_stages": a["arch_kwargs"]["n_stages"],
               "features_per_stage": a["arch_kwargs"]["features_per_stage"],
               "blocks_per_stage": a["arch_kwargs"].get("n_blocks_per_stage"),
               "normalization": c.get("normalization_schemes")}
    for k, v in summary.items():
        print(f"   {k:20s}: {v}")
    return summary


if STORE.done(PHASE_N):
    restore_preprocessed()
    PLAN_SUMMARY = print_plans()
else:
    # 1) raw dataset (symlinks into the Pass B cache)
    fetch_phase(PHASE_B, PASSB_DIR)
    raw_ds = NN_RAW / DS_NAME
    for sub in ["imagesTr", "labelsTr", "imagesTs", "labelsTs"]:
        (raw_ds / sub).mkdir(parents=True, exist_ok=True)
    n_tr = 0
    for _, r in rb.iterrows():
        tr = r.split in ("train", "val")
        for src, sub in [(PASSB_DIR / f"{r.study_id}_0000.nii.gz", "imagesTr" if tr else "imagesTs"),
                         (PASSB_DIR / f"{r.study_id}.nii.gz", "labelsTr" if tr else "labelsTs")]:
            dst = raw_ds / sub / src.name
            if not dst.exists():
                os.symlink(src, dst)
        n_tr += tr
    json.dump({"channel_names": {"0": "CT"}, "labels": {"background": 0, "pancreas": 1, "PDAC": 2},
               "numTraining": n_tr, "file_ending": ".nii.gz", "name": DS_NAME},
              open(raw_ds / "dataset.json", "w"), indent=2)
    print(f"Raw dataset: imagesTr {n_tr} | imagesTs {len(rb) - n_tr}")

    # 2) fingerprint + ResEnc-M plans + 3d_fullres preprocessing
    nn_cli("nnunetv2.experiment_planning.plan_and_preprocess_entrypoints", "plan_and_preprocess_entry",
           ["-d", DS_ID, "-pl", "nnUNetPlannerResEncM", "-c", "3d_fullres",
            "-np", max(1, (os.cpu_count() or 2) - 1), "-npfp", max(1, (os.cpu_count() or 2) - 1),
            "--verify_dataset_integrity"], "nnunet_plan_preprocess.log")

    # 3) our locked split as nnU-Net fold 0 (train = TRAIN split, val = VAL split; test never seen)
    json.dump([{"train": TRAIN_KEYS, "val": VAL_KEYS}], open(PRE_DS / "splits_final.json", "w"), indent=2)
    PLAN_SUMMARY = print_plans()
    pre_gb = sum(f.stat().st_size for f in PRE_DS.rglob("*") if f.is_file()) / 1e9
    print(f"Preprocessed size: {pre_gb:.1f} GB — uploading to the repo...")

    # 4) store for GPU sessions
    manifest = upload_preprocessed()
    STORE.mark_done(PHASE_N, {"plans": PLAN_SUMMARY, "n_train": len(TRAIN_KEYS), "n_val": len(VAL_KEYS),
                              "excluded_train": EXCLUDED_TRAIN, "preprocessed_gb": round(pre_gb, 1),
                              "parts": len(manifest["parts"])})
    print("✓ nnU-Net preprocessing COMPLETE and stored.")

Train 1557 | Val 337 | Test 338 | excluded from training (crop missed target): ['100028_00001', '100118_00001', '100852_00001', '100939_00001', '101303_00001', '101379_00001']
✓ Preprocessed data restored (7604 files) in 2.2 min
   patch_size          : [64, 128, 192]
   batch_size          : 2
   spacing             : [2.0, 1.0, 1.0]
   median_image_size   : [96.0, 224.0, 288.0]
   network             : ResidualEncoderUNet
   n_stages            : 6
   features_per_stage  : [32, 64, 128, 256, 320, 320]
   blocks_per_stage    : [1, 3, 4, 6, 6, 6]
   normalization       : ['CTNormalization']


In [12]:
# ============================================================
# CELL 12 — nnU-Net TRAINING (GPU session; resumable across sessions / accounts / platforms)
# Checkpoint pushed to the repo every 5 epochs; clean stop before the session limit.
# Just Save & Run All again (GPU) to continue — it resumes automatically.
# v2: no hang after the clean stop (data-loader workers are killed; watchdog before the 12 h limit)
# ============================================================
import torch
import nnunetv2
import threading, signal

PHASE_T = "nnunet_train"
TRAINER = "nnUNetTrainer_PANORAMA"
FOLD = 0
RUN_DIR = NN_RES / DS_NAME / f"{TRAINER}__{PLANS}__3d_fullres" / f"fold_{FOLD}"
STORE_RUN = f"{PHASE_T}/fold_{FOLD}"

if not STORE.done(PHASE_N):
    raise RuntimeError("nnU-Net preprocessing not complete.")

if STORE.done(PHASE_T):
    print("✓ nnU-Net training already complete — skipping.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU. Settings → Accelerator → 'GPU T4 x2', then Save & Run All again.")
    print(f"GPU: {torch.cuda.get_device_name(0)} | torch {torch.__version__} | CUDA {torch.version.cuda}")

    # ---- lock the training config (first session writes it; later sessions must reuse it) ----
    T_CONFIG = {"trainer": TRAINER, "plans": PLANS, "configuration": "3d_fullres", "fold": FOLD,
                "num_epochs": 300, "save_every": 5}
    stored = STORE.get_json(f"{PHASE_T}/config.json")
    if stored is None:
        STORE.put_json(T_CONFIG, f"{PHASE_T}/config.json", msg="LOCKED nnU-Net training config")
    else:
        T_CONFIG = stored
    print("Training config:", T_CONFIG)

    # ---- custom trainer = default nnU-Net + frequent HF checkpoints + clean time-limit stop ----
    TRAINER_SRC = r'''
import os, sys, json, time
from os.path import join, isfile
import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainer_PANORAMA(nnUNetTrainer):
    """Default nnU-Net recipe + configurable epochs, checkpoints pushed to a private HF repo,
    and a clean stop before the notebook session limit."""

    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict,
                 device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.num_epochs = int(os.environ["PDAC_NUM_EPOCHS"])
        self.save_every = int(os.environ["PDAC_SAVE_EVERY"])
        self.pdac_deadline = float(os.environ["PDAC_DEADLINE"])

    def _pdac_status(self):
        lg = getattr(self.logger, "my_fantastic_logging", {})
        def last(k):
            v = lg.get(k, [])
            return v[-1] if len(v) else None
        st, en = lg.get("epoch_start_timestamps", []), lg.get("epoch_end_timestamps", [])
        ep_time = (en[-1] - st[-1]) if len(st) and len(en) else None
        return {"epochs_done": int(self.current_epoch), "num_epochs": int(self.num_epochs),
                "train_loss": last("train_losses"), "val_loss": last("val_losses"),
                "pseudo_dice_per_class": last("dice_per_class_or_region"),
                "ema_fg_dice": last("ema_fg_dice"), "best_ema": getattr(self, "_best_ema", None),
                "epoch_time_s": ep_time, "time": time.strftime("%Y-%m-%d %H:%M:%S")}

    def _pdac_push(self):
        from huggingface_hub import HfApi, CommitOperationAdd
        api = HfApi(token=os.environ["HF_TOKEN"])
        repo, rtype, prefix = os.environ["PDAC_REPO"], os.environ["PDAC_REPO_TYPE"], os.environ["PDAC_STORE_RUN"]
        with open(join(self.output_folder, "status.json"), "w") as f:
            json.dump(self._pdac_status(), f, indent=2, default=float)
        names = ["checkpoint_final.pth", "checkpoint_latest.pth", "checkpoint_best.pth",
                 "progress.png", "debug.json", "status.json"]
        names += [f for f in os.listdir(self.output_folder) if f.startswith("training_log")]
        for attempt in range(6):
            try:
                ops = [CommitOperationAdd(path_in_repo=f"{prefix}/{n}", path_or_fileobj=join(self.output_folder, n))
                       for n in names if isfile(join(self.output_folder, n))]
                api.create_commit(repo_id=repo, repo_type=rtype, operations=ops,
                                  commit_message=f"nnU-Net epoch {self.current_epoch}")
                break
            except Exception as e:
                self.print_to_log_file(f"[PDAC] push retry {attempt + 1}: {e}")
                if attempt == 5:
                    raise
                time.sleep(20 * (attempt + 1))
        try:
            api.super_squash_history(repo_id=repo, repo_type=rtype)   # keeps repo storage small
        except Exception as e:
            self.print_to_log_file(f"[PDAC] history squash skipped: {e}")
        self.print_to_log_file(f"[PDAC] checkpoint pushed to HF at epoch {self.current_epoch}")

    def _pdac_hard_exit(self, code):
        """Kill the data-augmentation worker processes first, otherwise they keep the
        notebook's output pipe open and the cell hangs until Kaggle's 12 h kill."""
        try:
            import psutil
            kids = psutil.Process().children(recursive=True)
            for c in kids:
                try:
                    c.kill()
                except Exception:
                    pass
            psutil.wait_procs(kids, timeout=15)
        except Exception as e:
            print(f"[PDAC] worker cleanup issue: {e}")
        sys.stdout.flush(); sys.stderr.flush()
        os._exit(code)

    def on_epoch_end(self):
        super().on_epoch_end()                     # may write checkpoint_latest / checkpoint_best
        if self.local_rank != 0:
            return
        done = self.current_epoch                  # epochs completed so far
        if done % self.save_every == 0 and done < self.num_epochs:
            self._pdac_push()
        if time.time() > self.pdac_deadline and done < self.num_epochs:
            self.current_epoch -= 1                # save exactly like nnU-Net does inside on_epoch_end
            self.save_checkpoint(join(self.output_folder, "checkpoint_latest.pth"))
            self.current_epoch += 1
            self._pdac_push()
            with open(join(self.output_folder, "STOPPED_FOR_TIME"), "w") as f:
                f.write(str(done))
            self.print_to_log_file("[PDAC] session time budget reached — checkpoint pushed, stopping cleanly.")
            self._pdac_hard_exit(3)

    def on_train_end(self):
        super().on_train_end()                     # writes checkpoint_final.pth
        if self.local_rank == 0:
            self._pdac_push()

    def perform_actual_validation(self, save_probabilities: bool = False):
        if os.environ.get("PDAC_RUN_NNUNET_VAL") == "1":
            return super().perform_actual_validation(save_probabilities)
        self.print_to_log_file("[PDAC] skipping built-in validation (our own evaluation cell does it).")
'''
    tdir = Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer" / "variants" / "pdac"
    tdir.mkdir(parents=True, exist_ok=True)
    (tdir / "__init__.py").touch()
    (tdir / f"{TRAINER}.py").write_text(TRAINER_SRC)

    # ---- restore any previous progress from the repo ----
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    (RUN_DIR / "STOPPED_FOR_TIME").unlink(missing_ok=True)
    prev = STORE.list(STORE_RUN)
    for path in prev:
        p = Path(hf_hub_download(WORK_REPO, path, repo_type=WORK_REPO_TYPE, local_dir=str(SCRATCH / "dl"),
                                 token=HF_TOKEN, force_download=True))
        shutil.move(str(p), RUN_DIR / Path(path).name)
    prev_status = json.loads((RUN_DIR / "status.json").read_text()) if (RUN_DIR / "status.json").exists() else None
    print(f"Restored from repo: {[Path(p).name for p in prev] or 'nothing (fresh start)'}")
    if prev_status:
        print(f"Resuming after epoch {prev_status['epochs_done']} / {prev_status['num_epochs']} "
              f"| best EMA pseudo-Dice so far {prev_status.get('best_ema')}")

    # ---- launch ----
    deadline = KERNEL_START + (MAX_SESSION_HOURS - 0.6) * 3600       # trainer stops cleanly after this
    hard_stop = KERNEL_START + (MAX_SESSION_HOURS - 0.25) * 3600     # watchdog: kill if still running
    os.environ.update({
        "PDAC_NUM_EPOCHS": str(T_CONFIG["num_epochs"]), "PDAC_SAVE_EVERY": str(T_CONFIG["save_every"]),
        "PDAC_DEADLINE": str(deadline if MAX_SESSION_HOURS < 1e8 else 1e18),
        "PDAC_REPO": WORK_REPO, "PDAC_REPO_TYPE": WORK_REPO_TYPE, "PDAC_STORE_RUN": STORE_RUN,
        "HF_TOKEN": HF_TOKEN, "nnUNet_compile": "false",
        "nnUNet_n_proc_DA": str(os.cpu_count() or 4),
    })
    print(f"Session hours used {hours_used():.2f} | clean stop at ~{MAX_SESSION_HOURS - 0.6:.1f} h | "
          f"watchdog at ~{MAX_SESSION_HOURS - 0.25:.2f} h")
    code = ("import sys; sys.argv = ['nnUNetv2_train', '%d', '3d_fullres', '%d', '-tr', '%s', '-p', '%s', '--c']; "
            "from nnunetv2.run.run_training import run_training_entry; run_training_entry()"
            % (DS_ID, FOLD, TRAINER, PLANS))
    show = ("Epoch", "train_loss", "val_loss", "Pseudo dice", "Epoch time", "Yayy", "[PDAC]",
            "Error", "Traceback", "WARNING", "Using", "learning rate")
    log_path = WORK / "nnunet_train_stdout.log"
    t0 = time.time()
    proc = subprocess.Popen([sys.executable, "-c", code], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, env=os.environ.copy(),
                            start_new_session=True)          # own process group → we can kill all of it

    def _pump():
        with open(log_path, "a") as log:
            for line in proc.stdout:
                log.write(line)
                log.flush()
                if any(k in line for k in show) and "%|" not in line:
                    print(line.rstrip()[:200], flush=True)

    reader = threading.Thread(target=_pump, daemon=True)
    reader.start()

    watchdog_fired = False
    while proc.poll() is None:
        if time.time() > hard_stop:
            print("[PDAC] watchdog: training still running near the session limit — killing it. "
                  "The last checkpoint pushed to HF is kept.", flush=True)
            watchdog_fired = True
            break
        time.sleep(10)
    try:
        os.killpg(proc.pid, signal.SIGKILL)               # also kills orphaned data-loader workers
    except (ProcessLookupError, PermissionError):
        pass
    rc = proc.wait()
    reader.join(timeout=60)
    print(f"\nTraining process exited with code {rc} after {(time.time() - t0) / 3600:.2f} h")

    st = json.loads((RUN_DIR / "status.json").read_text()) if (RUN_DIR / "status.json").exists() else None
    if (RUN_DIR / "checkpoint_final.pth").exists() and rc == 0:
        STORE.mark_done(PHASE_T, st)
        print("✓ nnU-Net TRAINING COMPLETE:", st)
    elif rc == 3 or (RUN_DIR / "STOPPED_FOR_TIME").exists() or watchdog_fired:
        print(f"⏸ Paused safely at epoch {st['epochs_done'] if st else '?'}/{T_CONFIG['num_epochs']} "
              f"— run Save & Run All (GPU) again to continue.")
        print("Status:", st)
    else:
        print("".join(open(log_path).readlines()[-80:]))
        raise RuntimeError("Training stopped with an error — see the lines above.")

GPU: Tesla T4 | torch 2.11.0+cu128 | CUDA 12.8
Training config: {'trainer': 'nnUNetTrainer_PANORAMA', 'plans': 'nnUNetResEncUNetMPlans', 'configuration': '3d_fullres', 'fold': 0, 'num_epochs': 300, 'save_every': 5}
Restored from repo: ['checkpoint_best.pth', 'checkpoint_latest.pth', 'debug.json', 'progress.png', 'status.json', 'training_log_2026_10_19_19_40_10.txt', 'training_log_2026_10_20_10_49_00.txt']
Resuming after epoch 259 / 300 | best EMA pseudo-Dice so far 0.7730979919433594
Session hours used 0.06 | clean stop at ~10.9 h | watchdog at ~11.25 h
Using device: cuda:0
2026-10-21 03:39:02.587215: Using splits from existing split file: /kaggle/temp/pdac_scratch/nnUNet_preprocessed/Dataset501_PANORAMA/splits_final.json
2026-10-21 03:39:08.843712: Epoch 259
2026-10-21 03:39:08.844095: Current learning rate: 0.00167
2026-10-21 03:43:55.854624: train_loss -0.5384
2026-10-21 03:43:55.855245: val_loss -0.519
2026-10-21 03:43:55.855523: Pseudo dice [np.float32(0.9271), np.float32(0.529)]


In [13]:
# ============================================================
# CELL 13 — FINAL EVALUATION + FIGURES (pre-registered; test set scored once)
# 1) nnU-Net inference with checkpoint_final on VAL + TEST ROI crops (GPU, resumable, stored in repo)
# 2) threshold + post-processing chosen on VALIDATION ONLY by the locked rule below
# 3) single test evaluation: gold (manual) + silver (automatic) metrics, patient-level detection
# 4) patient-level bootstrap 95% CIs + figures (shown here, saved to Output, uploaded to eval/)
# Re-run after any interruption — it resumes. Only step 1 needs a GPU.
# ============================================================
import warnings
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import ndimage
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
import nnunetv2

warnings.filterwarnings("ignore", category=RuntimeWarning)
PHASE_I, PHASE_E = "eval_infer", "eval"
EVAL_DIR = SCRATCH / "eval_pred"
OUT_DIR = WORK / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

if not STORE.done(PHASE_T):
    raise RuntimeError("nnU-Net training is not complete yet — let cell 12 finish (epoch 300) first.")

# ---------------- PRE-REGISTERED PROTOCOL (locked in the repo before any metric is computed) ----------------
PROTOCOL = {
    "checkpoint": "checkpoint_final.pth (epoch 300)",
    "inference": {"tile_step_size": 0.5, "gaussian": True, "mirroring": True},
    "evaluation_space": "CT-only ROI crop 96x224x288 at 2x1x1 mm (z,y,x)",
    "tumour_mask": "PDAC softmax probability >= threshold, 26-connected components",
    "thresholds": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    "min_lesion_mm3": [0, 100, 250, 500, 1000],
    "keep_largest_only": [False, True],
    "selection_rule": ("maximise mean tumour Dice on manual (gold) VAL PDAC cases subject to "
                       "false-positive rate <= 0.15 on VAL non-PDAC scans; if no configuration meets it, "
                       "take the lowest false-positive rate (ties: higher Dice)"),
    "fp_rate_max": 0.15,
    "patient_score": "maximum PDAC probability in the ROI",
    "patient_positive": "at least one lesion kept after post-processing",
    "bootstrap_resamples": 2000,
    "bootstrap_unit": "patient",
    "seed": SEED,
}
stored = STORE.get_json(f"{PHASE_E}/protocol.json")
if stored is None:
    STORE.put_json(PROTOCOL, f"{PHASE_E}/protocol.json", msg="LOCKED evaluation protocol (pre-registered)")
    print("Protocol written and LOCKED.")
elif stored != PROTOCOL:
    raise RuntimeError("The evaluation protocol differs from the locked one in the repo — not allowed.")
else:
    print("Protocol loaded (LOCKED).")

EVAL_COHORT = cohort_B[cohort_B.split.isin(["val", "test"])].reset_index(drop=True)
print(pd.crosstab([EVAL_COHORT.split, EVAL_COHORT.tier], EVAL_COHORT.label, margins=True))

# ---------------- inputs: only the Pass B shards that hold val/test crops ----------------
def fetch_passB_cases(study_ids):
    idx = STORE.get_json(f"{PHASE_B}/index.json")
    need = sorted({idx["done"][s]["shard"] for s in study_ids})
    PASSB_DIR.mkdir(parents=True, exist_ok=True)
    t0, n_new = time.time(), 0
    for name in need:
        if (PASSB_DIR / f".{name}.ok").exists():
            continue
        p = Path(hf_hub_download(WORK_REPO, f"{PHASE_B}/shards/{name}.tar", repo_type=WORK_REPO_TYPE,
                                 local_dir=str(SCRATCH / "dl"), token=HF_TOKEN))
        with tarfile.open(p) as tar:
            tar.extractall(PASSB_DIR, filter="data")
        p.unlink(missing_ok=True)
        (PASSB_DIR / f".{name}.ok").touch()
        n_new += 1
    missing = [s for s in study_ids if not (PASSB_DIR / f"{s}_0000.nii.gz").exists()
               or not (PASSB_DIR / f"{s}.nii.gz").exists()]
    if missing:
        raise RuntimeError(f"{len(missing)} val/test crops missing after download, e.g. {missing[:5]}")
    print(f"✓ Val/test crops ready ({len(study_ids)} cases, {len(need)} shards, {n_new} downloaded "
          f"in {(time.time() - t0) / 60:.1f} min)")

fetch_passB_cases(EVAL_COHORT.study_id.tolist())

# ---------------- final checkpoint + training curves ----------------
MODEL_DIR = SCRATCH / "eval_model" / f"{TRAINER}__{PLANS}__3d_fullres"

def get_final_checkpoint():
    ck = MODEL_DIR / f"fold_{FOLD}" / "checkpoint_final.pth"
    if not ck.exists():
        ck.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(STORE.get(f"{STORE_RUN}/checkpoint_final.pth", force=True)), ck)
    return ck

if not STORE.exists(f"{PHASE_E}/training_curves.json"):
    lg = torch.load(get_final_checkpoint(), map_location="cpu", weights_only=False)["logging"]
    tc = {k: ([[float(x) for x in np.atleast_1d(v)] for v in lg[k]] if k == "dice_per_class_or_region"
              else [float(v) for v in lg[k]]) for k in lg}
    STORE.put_json(tc, f"{PHASE_E}/training_curves.json", msg="training curves from checkpoint_final")
TRAIN_CURVES = STORE.get_json(f"{PHASE_E}/training_curves.json")

# ---------------- step 1: inference (GPU, resumable) ----------------
if not STORE.done(PHASE_I):
    if not torch.cuda.is_available():
        raise RuntimeError("Inference needs a GPU: Settings → Accelerator → 'GPU T4 x2', then Save & Run All.")
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO

    (MODEL_DIR / f"fold_{FOLD}").mkdir(parents=True, exist_ok=True)
    shutil.copy(PRE_DS / f"{PLANS}.json", MODEL_DIR / "plans.json")
    shutil.copy(PRE_DS / "dataset.json", MODEL_DIR / "dataset.json")
    get_final_checkpoint()
    tdir = Path(nnunetv2.__file__).parent / "training" / "nnUNetTrainer" / "variants" / "pdac"
    tdir.mkdir(parents=True, exist_ok=True)
    (tdir / "__init__.py").touch()
    if not (tdir / f"{TRAINER}.py").exists():      # class must be importable to rebuild the network
        (tdir / f"{TRAINER}.py").write_text(
            "from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\n\n\n"
            f"class {TRAINER}(nnUNetTrainer):\n    pass\n")

    PREDICTOR = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
                                perform_everything_on_device=True, device=torch.device("cuda", 0),
                                verbose=False, verbose_preprocessing=False, allow_tqdm=False)
    PREDICTOR.initialize_from_trained_model_folder(str(MODEL_DIR), use_folds=(FOLD,),
                                                   checkpoint_name="checkpoint_final.pth")
    SIO = SimpleITKIO()

    def infer_worker(row, out_dir):
        t0 = time.time()
        sid = row["study_id"]
        img, props = SIO.read_images([str(PASSB_DIR / f"{sid}_0000.nii.gz")])
        _, prob = PREDICTOR.predict_single_npy_array(img, props, None, None, True)
        if hasattr(prob, "cpu"):
            prob = prob.cpu().numpy()
        prob = np.asarray(prob, dtype=np.float32)
        q = lambda a: np.round(np.clip(a, 0, 1) * 255).astype(np.uint8)
        f = Path(out_dir) / f"{sid}.npz"
        np.savez_compressed(f, p_panc=q(prob[1]), p_pdac=q(prob[2]))
        return {"study_id": sid, "shape": list(prob.shape[1:]), "seconds": round(time.time() - t0, 1)}, [f]

    _nw = N_WORKERS
    N_WORKERS = 1                                  # one GPU predictor → sequential
    try:
        index_I = run_stream_phase(PHASE_I, EVAL_COHORT, infer_worker, EVAL_DIR, shard_size=25)
    finally:
        N_WORKERS = _nw
    left = set(EVAL_COHORT.study_id) - set(index_I["done"])
    if not left:
        STORE.mark_done(PHASE_I, {"n": len(index_I["done"])})
        print("✓ Inference COMPLETE for all val + test cases.")
    else:
        print(f"⏸ Inference not finished ({len(left)} cases left). Save & Run All (GPU) again to resume.")

# ---------------- steps 2–4 run only when all predictions exist ----------------
if STORE.done(PHASE_I):
    fetch_phase(PHASE_I, EVAL_DIR)
    STRUCT26 = ndimage.generate_binary_structure(3, 3)
    STRUCT6 = ndimage.generate_binary_structure(3, 1)
    SPACING = tuple(float(s) for s in FINE_SPACING_ZYX)        # (2, 1, 1) mm
    VOX_MM3 = float(np.prod(FINE_SPACING_ZYX))
    THR = PROTOCOL["thresholds"]
    THR_U8 = [int(round(t * 255)) for t in THR]
    rng = np.random.default_rng(SEED)

    def load_case(sid):
        with np.load(EVAL_DIR / f"{sid}.npz") as d:
            pp, pt = d["p_panc"], d["p_pdac"]
        gt = sitk.GetArrayFromImage(sitk.ReadImage(str(PASSB_DIR / f"{sid}.nii.gz")))
        if gt.shape != pt.shape:
            raise RuntimeError(f"{sid}: prediction {pt.shape} vs label {gt.shape}")
        return pp, pt, gt

    def bbox(mask, margin=2):
        idx = np.argwhere(mask)
        lo = np.maximum(idx.min(0) - margin, 0)
        hi = np.minimum(idx.max(0) + margin + 1, mask.shape)
        return tuple(slice(a, b) for a, b in zip(lo, hi))

    def components_all_thresholds(pt, gt_t):
        base = (pt >= min(THR_U8)) | gt_t
        empty = (np.zeros(0, np.int64), np.zeros(0, np.int64))
        if not base.any():
            return [empty] * len(THR_U8)
        sl = bbox(base)
        ptc, gtc = pt[sl], gt_t[sl]
        out = []
        for u in THR_U8:
            m = ptc >= u
            if not m.any():
                out.append(empty)
                continue
            lab, n = ndimage.label(m, structure=STRUCT26)
            out.append((np.bincount(lab.ravel(), minlength=n + 1)[1:],
                        np.bincount(lab[gtc].ravel(), minlength=n + 1)[1:]))
        return out

    def apply_cfg(stats, ti, min_vox, largest):
        sizes, ov = stats["comp"][ti]
        keep = sizes >= max(min_vox, 1)
        if largest and keep.any():
            j = int(np.argmax(np.where(keep, sizes, -1)))
            keep = np.zeros(len(sizes), bool)
            keep[j] = True
        ps, inter, g = int(sizes[keep].sum()), int(ov[keep].sum()), stats["gt_n"]
        return (2 * inter / (ps + g) if ps + g > 0 else 1.0), bool(keep.any())

    def final_mask(pt, cfg):
        m = pt >= int(round(cfg["threshold"] * 255))
        if not m.any():
            return m
        lab, n = ndimage.label(m, structure=STRUCT26)
        sizes = np.bincount(lab.ravel(), minlength=n + 1)
        keep = sizes >= max(int(np.ceil(cfg["min_lesion_mm3"] / VOX_MM3)), 1)
        keep[0] = False
        if cfg["keep_largest"] and keep.any():
            j = int(np.argmax(np.where(keep, sizes, -1)))
            keep[:] = False
            keep[j] = True
        return keep[lab]

    def dice(a, b):
        s = a.sum() + b.sum()
        return float(2 * np.logical_and(a, b).sum() / s) if s > 0 else 1.0

    def hd95(a, b):
        if not a.any() or not b.any():
            return np.nan
        sl = bbox(a | b, margin=6)
        a, b = a[sl], b[sl]
        sa = a ^ ndimage.binary_erosion(a, STRUCT6)
        sb = b ^ ndimage.binary_erosion(b, STRUCT6)
        da = ndimage.distance_transform_edt(~sb, sampling=SPACING)[sa]
        db = ndimage.distance_transform_edt(~sa, sampling=SPACING)[sb]
        return float(np.percentile(np.concatenate([da, db]), 95))

    # ---------------- step 2: selection on VALIDATION only ----------------
    val = EVAL_COHORT[EVAL_COHORT.split == "val"].reset_index(drop=True)
    t0 = time.time()
    VAL_STATS = {}
    for i, sid in enumerate(val.study_id):
        _, pt, gt = load_case(sid)
        gt_t = gt == 2
        VAL_STATS[sid] = {"gt_n": int(gt_t.sum()), "comp": components_all_thresholds(pt, gt_t)}
        if (i + 1) % 50 == 0:
            print(f"   val component analysis {i + 1}/{len(val)} | {(time.time() - t0) / 60:.1f} min")
    gold_val = val[(val.tier == "manual") & (val.label == "PDAC")].study_id.tolist()
    healthy_val = val[val.label != "PDAC"].study_id.tolist()
    rows = []
    for ti, t in enumerate(THR):
        for mm in PROTOCOL["min_lesion_mm3"]:
            mv = int(np.ceil(mm / VOX_MM3))
            for lgst in PROTOCOL["keep_largest_only"]:
                d = [apply_cfg(VAL_STATS[s], ti, mv, lgst)[0] for s in gold_val]
                fp = [apply_cfg(VAL_STATS[s], ti, mv, lgst)[1] for s in healthy_val]
                rows.append({"threshold": t, "min_lesion_mm3": mm, "keep_largest": lgst,
                             "val_gold_mean_dice": float(np.mean(d)), "val_gold_median_dice": float(np.median(d)),
                             "val_fp_rate": float(np.mean(fp))})
    grid = pd.DataFrame(rows)
    grid.to_csv(OUT_DIR / "val_grid.csv", index=False)
    ok = grid[grid.val_fp_rate <= PROTOCOL["fp_rate_max"]]
    best = (ok.sort_values(["val_gold_mean_dice", "val_fp_rate"], ascending=[False, True]).iloc[0] if len(ok)
            else grid.sort_values(["val_fp_rate", "val_gold_mean_dice"], ascending=[True, False]).iloc[0])
    choice = {"threshold": float(best.threshold), "min_lesion_mm3": int(best.min_lesion_mm3),
              "keep_largest": bool(best.keep_largest), "val_gold_mean_dice": float(best.val_gold_mean_dice),
              "val_fp_rate": float(best.val_fp_rate), "constraint_met": bool(len(ok) > 0),
              "n_val_gold": len(gold_val), "n_val_healthy": len(healthy_val)}
    stored_cfg = STORE.get_json(f"{PHASE_E}/selected_config.json")
    if stored_cfg is None:
        STORE.put_json(choice, f"{PHASE_E}/selected_config.json", msg="LOCKED config selected on validation")
        CFG = choice
        print("Configuration selected on validation and LOCKED:", CFG)
    else:
        CFG = stored_cfg
        print("Using LOCKED configuration from the repo:", CFG)

    # ---------------- step 3: single TEST evaluation ----------------
    test = EVAL_COHORT[EVAL_COHORT.split == "test"].reset_index(drop=True)
    recs = []
    t0 = time.time()
    for i, r in test.iterrows():
        pp, pt, gt = load_case(r.study_id)
        gt_t, pred_t = gt == 2, final_mask(pt, CFG)
        pb = 255 - pp.astype(np.int16) - pt.astype(np.int16)
        pred_panc = np.maximum(pp, pt).astype(np.int16) > pb          # = argmax != background
        has_t = bool(gt_t.any())
        recs.append({
            "study_id": r.study_id, "patient_id": r.patient_id, "tier": r.tier, "label": r.label,
            "y_pdac": int(r.label == "PDAC"), "score": float(pt.max()) / 255.0,
            "pred_positive": int(pred_t.any()),
            "gt_tumour_ml": gt_t.sum() * VOX_MM3 / 1000, "pred_tumour_ml": pred_t.sum() * VOX_MM3 / 1000,
            "tumour_dice": dice(gt_t, pred_t) if has_t else np.nan,
            "tumour_detected": int(np.logical_and(gt_t, pred_t).any()) if has_t else np.nan,
            "tumour_hd95_mm": hd95(gt_t, pred_t) if has_t else np.nan,
            "pancreas_dice": dice(gt > 0, pred_panc) if (gt > 0).any() else np.nan,
        })
        if (i + 1) % 50 == 0:
            print(f"   test evaluation {i + 1}/{len(test)} | {(time.time() - t0) / 60:.1f} min")
    per_case = pd.DataFrame(recs)
    per_case.to_csv(OUT_DIR / "test_per_case.csv", index=False)

    # ---------------- step 4: patient-level bootstrap ----------------
    def pboot(df, fn):
        df = df.reset_index(drop=True)
        point = float(fn(df))
        pats = df.patient_id.values
        groups = [np.flatnonzero(pats == p) for p in pd.unique(pats)]
        vals = []
        for _ in range(PROTOCOL["bootstrap_resamples"]):
            idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
            try:
                v = float(fn(df.iloc[idx]))
                if np.isfinite(v):
                    vals.append(v)
            except ValueError:
                pass
        lo, hi = np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)
        return {"value": point, "ci_low": float(lo), "ci_high": float(hi), "n": int(len(df))}

    gold = per_case[(per_case.tier == "manual") & (per_case.y_pdac == 1)]
    silver = per_case[(per_case.tier == "automatic") & (per_case.y_pdac == 1)]
    healthy = per_case[per_case.y_pdac == 0]
    gold_hd = gold[gold.tumour_hd95_mm.notna()]
    R = {
        "Tumour Dice — gold test, mean": pboot(gold, lambda d: d.tumour_dice.mean()),
        "Tumour Dice — gold test, median": pboot(gold, lambda d: d.tumour_dice.median()),
        "Tumour detection sensitivity — gold test": pboot(gold, lambda d: d.tumour_detected.mean()),
        "Tumour HD95 (mm) — gold test, median*": pboot(gold_hd, lambda d: d.tumour_hd95_mm.median()),
        "Pancreas Dice — gold test, mean†": pboot(gold, lambda d: d.pancreas_dice.mean()),
        "Tumour Dice — silver test, mean": pboot(silver, lambda d: d.tumour_dice.mean()),
        "Patient sensitivity (all PDAC test)": pboot(per_case[per_case.y_pdac == 1], lambda d: d.pred_positive.mean()),
        "Patient specificity (non-PDAC test)": pboot(healthy, lambda d: 1 - d.pred_positive.mean()),
        "Patient AUROC (all test)": pboot(per_case, lambda d: roc_auc_score(d.y_pdac, d.score)),
        "Patient average precision (all test)": pboot(per_case, lambda d: average_precision_score(d.y_pdac, d.score)),
    }
    results = {"config": CFG, "metrics": R, "protocol": PROTOCOL,
               "notes": ["Evaluation in the CT-only ROI crop at 2x1x1 mm; every test tumour lies fully inside the crop.",
                         "* HD95 over cases where the model predicted a lesion.",
                         "† Reference pancreas masks are algorithm-generated in PANORAMA (also for manual cases).",
                         "Silver = automatic (algorithm-generated) tumour labels.",
                         "95% CIs: bootstrap over patients, 2000 resamples."]}
    (OUT_DIR / "test_results.json").write_text(json.dumps(results, indent=2))

    print("\n" + "=" * 92)
    print(f"LOCKED config: threshold {CFG['threshold']} | min lesion {CFG['min_lesion_mm3']} mm³ | "
          f"keep largest {CFG['keep_largest']}")
    print(f"{'Metric':52s} {'n':>5s} {'value':>8s}   95% CI")
    for k, v in R.items():
        print(f"{k:52s} {v['n']:5d} {v['value']:8.3f}   [{v['ci_low']:.3f}, {v['ci_high']:.3f}]")
    for n in results["notes"]:
        print("  ", n)
    print("=" * 92)

    # ---------------- figures ----------------
    figs = []
    def save(fig, name):
        fig.tight_layout()
        p = OUT_DIR / name
        fig.savefig(p, bbox_inches="tight")
        figs.append(p)
        plt.show()
        plt.close(fig)

    # Fig 1 — training curves
    tc = TRAIN_CURVES
    dpc = np.array(tc["dice_per_class_or_region"], dtype=float)
    n = min(len(tc["train_losses"]), len(tc["val_losses"]), len(dpc), len(tc["ema_fg_dice"]), len(tc["lrs"]))
    e = np.arange(n)
    st = np.array(tc.get("epoch_start_timestamps", []), dtype=float)
    breaks = list(np.where(np.diff(st[:n]) > 1800)[0] + 1) if len(st) >= n else []
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    ax[0].plot(e, tc["train_losses"][:n], label="train"); ax[0].plot(e, tc["val_losses"][:n], label="validation")
    ax[0].set(title="Loss (Dice + CE)", xlabel="epoch", ylabel="loss")
    ax[1].plot(e, dpc[:n, 0], alpha=.35, color="C0", label="pancreas (per epoch)")
    ax[1].plot(e, dpc[:n, 1], alpha=.35, color="C3", label="PDAC (per epoch)")
    ax[1].plot(e, tc["ema_fg_dice"][:n], color="k", lw=2, label="EMA mean (model selection)")
    ax[1].set(title="Validation pseudo-Dice (patch-based)", xlabel="epoch", ylabel="Dice", ylim=(0, 1))
    ax[2].plot(e, tc["lrs"][:n], color="C2"); ax[2].set(title="Learning rate (poly)", xlabel="epoch")
    for a in ax:
        for b in breaks:
            a.axvline(b, color="grey", ls=":", lw=1)
    ax[0].legend(); ax[1].legend(fontsize=8, loc="lower right")
    fig.suptitle("nnU-Net ResEnc-M training — dotted lines = Kaggle session restarts", y=1.03)
    save(fig, "fig1_training_curves.png")

    # Fig 2 — validation selection
    g2 = grid[grid.keep_largest == CFG["keep_largest"]]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    for mm, gg in g2.groupby("min_lesion_mm3"):
        ax[0].plot(gg.threshold, gg.val_gold_mean_dice, marker="o", label=f"min {mm} mm³")
        ax[1].plot(gg.threshold, gg.val_fp_rate, marker="o", label=f"min {mm} mm³")
    sel = g2[(g2.threshold == CFG["threshold"]) & (g2.min_lesion_mm3 == CFG["min_lesion_mm3"])]
    ax[0].plot(sel.threshold, sel.val_gold_mean_dice, "k*", ms=16, label="selected")
    ax[1].plot(sel.threshold, sel.val_fp_rate, "k*", ms=16, label="selected")
    ax[1].axhline(PROTOCOL["fp_rate_max"], color="r", ls="--", lw=1, label="pre-registered FP limit")
    ax[0].set(title=f"VAL: mean tumour Dice (gold, n={len(gold_val)})", xlabel="probability threshold", ylabel="Dice")
    ax[1].set(title=f"VAL: false-positive rate (non-PDAC, n={len(healthy_val)})", xlabel="probability threshold",
              ylabel="fraction of scans with a lesion")
    ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
    fig.suptitle(f"Post-processing chosen on validation only (keep largest = {CFG['keep_largest']})", y=1.03)
    save(fig, "fig2_validation_selection.png")

    # Fig 3 — test Dice distributions
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.2, 1]})
    data = [gold.tumour_dice.values, silver.tumour_dice.values, gold.pancreas_dice.dropna().values]
    names = [f"Tumour\ngold (n={len(gold)})", f"Tumour\nsilver (n={len(silver)})", f"Pancreas†\ngold (n={len(gold)})"]
    ax[0].boxplot(data, widths=.5, showfliers=False, medianprops={"color": "k"})
    for k, dd in enumerate(data, 1):
        ax[0].scatter(k + rng.uniform(-.15, .15, len(dd)), dd, s=14, alpha=.6, color=["C3", "C1", "C0"][k - 1])
    ax[0].set_xticks([1, 2, 3], names); ax[0].set(ylabel="Dice", ylim=(-.02, 1.02), title="TEST: per-case Dice")
    ax[1].hist(gold.tumour_dice, bins=np.linspace(0, 1, 21), color="C3", alpha=.8, edgecolor="white")
    m, md = R["Tumour Dice — gold test, mean"], R["Tumour Dice — gold test, median"]
    ax[1].axvline(m["value"], color="k", lw=2, label=f"mean {m['value']:.3f} [{m['ci_low']:.3f}, {m['ci_high']:.3f}]")
    ax[1].axvline(md["value"], color="k", ls="--", lw=2, label=f"median {md['value']:.3f}")
    ax[1].set(title=f"TEST gold tumour Dice (n={len(gold)})", xlabel="Dice", ylabel="cases"); ax[1].legend(fontsize=8)
    save(fig, "fig3_test_dice_distribution.png")

    # Fig 4 — size dependence and volume agreement
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
    det = gold.tumour_detected == 1
    ax[0].scatter(gold.gt_tumour_ml[det], gold.tumour_dice[det], s=22, color="C3", label="detected")
    ax[0].scatter(gold.gt_tumour_ml[~det], gold.tumour_dice[~det], s=40, marker="x", color="k", label="missed")
    ax[0].set(xscale="log", xlabel="true tumour volume (mL, log)", ylabel="Dice", ylim=(-.02, 1.02),
              title="TEST gold: Dice vs tumour size"); ax[0].legend(fontsize=8)
    floor = 0.01
    pv = gold.pred_tumour_ml.clip(lower=floor)
    ax[1].scatter(gold.gt_tumour_ml, pv, s=22, color="C3")
    lim = [floor, max(gold.gt_tumour_ml.max(), pv.max()) * 1.5]
    ax[1].plot(lim, lim, "k--", lw=1, label="identity")
    ax[1].set(xscale="log", yscale="log", xlim=lim, ylim=lim, xlabel="true volume (mL)", ylabel="predicted volume (mL)",
              title=f"Volume agreement (no prediction plotted at {floor} mL)"); ax[1].legend(fontsize=8)
    save(fig, "fig4_size_and_volume.png")

    # Fig 5 — ROC and PR (patient level)
    fpr, tpr, _ = roc_curve(per_case.y_pdac, per_case.score)
    prec, rec_, _ = precision_recall_curve(per_case.y_pdac, per_case.score)
    au, ap = R["Patient AUROC (all test)"], R["Patient average precision (all test)"]
    se, sp = R["Patient sensitivity (all PDAC test)"], R["Patient specificity (non-PDAC test)"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
    ax[0].plot(fpr, tpr, color="C3", lw=2, label=f"AUROC {au['value']:.3f} [{au['ci_low']:.3f}, {au['ci_high']:.3f}]")
    ax[0].plot([0, 1], [0, 1], "k:", lw=1)
    ax[0].plot(1 - sp["value"], se["value"], "ko", ms=8, label="operating point (selected config)")
    ax[0].set(xlabel="1 − specificity", ylabel="sensitivity", title=f"TEST patient-level ROC (n={len(per_case)})")
    ax[0].legend(fontsize=8, loc="lower right")
    ax[1].plot(rec_, prec, color="C0", lw=2, label=f"AP {ap['value']:.3f} [{ap['ci_low']:.3f}, {ap['ci_high']:.3f}]")
    ax[1].axhline(per_case.y_pdac.mean(), color="k", ls=":", lw=1, label=f"prevalence {per_case.y_pdac.mean():.2f}")
    ax[1].set(xlabel="recall", ylabel="precision", title="TEST precision–recall"); ax[1].legend(fontsize=8)
    save(fig, "fig5_roc_pr.png")

    # Fig 6 — confusion matrix
    tp = int(((per_case.y_pdac == 1) & (per_case.pred_positive == 1)).sum())
    fn_ = int(((per_case.y_pdac == 1) & (per_case.pred_positive == 0)).sum())
    fp_ = int(((per_case.y_pdac == 0) & (per_case.pred_positive == 1)).sum())
    tn = int(((per_case.y_pdac == 0) & (per_case.pred_positive == 0)).sum())
    cm = np.array([[tp, fn_], [fp_, tn]])
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ax.imshow(cm / cm.sum(axis=1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
    for (a, b), v in np.ndenumerate(cm):
        ax.text(b, a, f"{v}\n({v / cm[a].sum():.0%})", ha="center", va="center", fontsize=12)
    ax.set_xticks([0, 1], ["predicted PDAC", "predicted no PDAC"])
    ax.set_yticks([0, 1], ["PDAC", "non-PDAC"])
    ax.set_title(f"TEST patient-level (n={len(per_case)})")
    ax.spines[:].set_visible(False)
    save(fig, "fig6_confusion_matrix.png")

    # Fig 7 — example cases (best / median / worst gold Dice)
    gs = gold.sort_values("tumour_dice").reset_index(drop=True)
    picks = [("best", gs.iloc[-1]), ("median", gs.iloc[(gs.tumour_dice - gs.tumour_dice.median()).abs().idxmin()]),
             ("worst", gs.iloc[0])]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5.2))
    for a, (tag, r) in zip(ax, picks):
        _, pt, gt = load_case(r.study_id)
        hu = sitk.GetArrayFromImage(sitk.ReadImage(str(PASSB_DIR / f"{r.study_id}_0000.nii.gz")))
        gt_t, pred_t = gt == 2, final_mask(pt, CFG)
        z = int(np.argmax(gt_t.sum(axis=(1, 2))))
        cy, cx = np.argwhere(gt_t[z]).mean(axis=0).astype(int)
        h = 60
        y0, x0 = max(cy - h, 0), max(cx - h, 0)
        sl = (slice(y0, y0 + 2 * h), slice(x0, x0 + 2 * h))
        a.imshow(np.clip(hu[z][sl], -100, 400), cmap="gray")
        a.contour(gt_t[z][sl], levels=[.5], colors="lime", linewidths=2)
        if pred_t[z][sl].any():
            a.contour(pred_t[z][sl], levels=[.5], colors="red", linewidths=2)
        a.set_title(f"{tag}: Dice {r.tumour_dice:.2f} | {r.gt_tumour_ml:.1f} mL")
        a.axis("off")
    fig.legend(handles=[Line2D([], [], color="lime", lw=2, label="ground truth (manual)"),
                        Line2D([], [], color="red", lw=2, label="prediction")],
               loc="lower center", ncol=2, frameon=False)
    fig.suptitle("TEST gold cases — axial slice with the largest tumour area (window −100…400 HU)", y=1.0)
    save(fig, "fig7_examples.png")

    # Fig 8 — results table
    fig, ax = plt.subplots(figsize=(11, 0.45 * len(R) + 1.4))
    ax.axis("off")
    cells = [[k, v["n"], f"{v['value']:.3f}", f"[{v['ci_low']:.3f}, {v['ci_high']:.3f}]"] for k, v in R.items()]
    tbl = ax.table(cellText=cells, colLabels=["Metric", "n", "Value", "95% CI (patient bootstrap)"],
                   loc="center", cellLoc="left", colWidths=[.55, .07, .12, .26])
    tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1, 1.5)
    ax.set_title(f"PANORAMA PDAC — held-out test results (config locked on validation: thr {CFG['threshold']}, "
                 f"min {CFG['min_lesion_mm3']} mm³, largest {CFG['keep_largest']})", fontsize=10)
    fig.text(0.01, 0.01, "  ".join(results["notes"][1:3]), fontsize=7.5)
    save(fig, "fig8_results_table.png")

    # ---------------- store everything ----------------
    pairs = [(p, f"{PHASE_E}/figures/{p.name}") for p in figs]
    pairs += [(OUT_DIR / f, f"{PHASE_E}/{f}") for f in ["test_results.json", "test_per_case.csv", "val_grid.csv"]]
    commit_files(pairs, msg="evaluation results + figures")
    if not STORE.done(PHASE_E):
        STORE.mark_done(PHASE_E, {k: round(v["value"], 4) for k, v in R.items()})
    print(f"\n✓ EVALUATION COMPLETE — figures + tables in {OUT_DIR} (Output tab) and {WORK_REPO}/{PHASE_E}/")

Protocol written and LOCKED.
label            PDAC  non-PDAC  All
split tier                          
test  automatic    29       237  266
      manual       72         0   72
val   automatic    30       233  263
      manual       73         1   74
All               204       471  675
✓ Val/test crops ready (675 cases, 14 shards, 14 downloaded in 0.9 min)
[eval_infer] cohort 675 | done 0 | excluded 0 | pending 675 | workers 1 | session hours used 3.28
   25/675 | ok 25 fail 0 | 3.6 cases/min | ETA 3.02 h | RAM 24%
   ↑ saved shard_0000 | total done 25
   50/675 | ok 50 fail 0 | 3.6 cases/min | ETA 2.89 h | RAM 24%
   ↑ saved shard_0001 | total done 50
   75/675 | ok 75 fail 0 | 3.6 cases/min | ETA 2.77 h | RAM 24%
   ↑ saved shard_0002 | total done 75
   100/675 | ok 100 fail 0 | 3.6 cases/min | ETA 2.65 h | RAM 24%
   ↑ saved shard_0003 | total done 100
   125/675 | ok 125 fail 0 | 3.6 cases/min | ETA 2.54 h | RAM 24%
   ↑ saved shard_0004 | total done 125
   150/675 | ok 150 fail 0